# Fairness Analysis Methods Template

## Context and Objective

The University of California, Davis Health system (UCDH) is committed to ensuring that clinical prediction models used in the health system do not perpetuate or exacerbate health disparities. As part of fulfilling this commitment, prediction models proposed for clinical use are evaluated for bias and fairness concerns.

UCDH developed an approach to evaluating fairness of binary clinical prediction models as documented in Taylor et al. (forthcoming), *From Performance Metrics to Patient Consequences: A Framework for Assessing Fairness in Clinical Prediction Models*. This Python template conducts statistical analyses for a binary prediction model in support of Phase 2: evaluate performance differences using the UCDH fairness assessment approach.  

Specifically, this template:

1. describes model performance for priority attribute groups: race/ethnicity, sex or gender, age, and Healthy Places Index (HPI);
2. conducts analyses to identify statistically significant performance differences across those groups; and
3. quantifies the magnitude of performance differences and identifies potentially important differences.


## Getting Started

This template is for binary outcomes. **The first configuration code cell is the only cell users need to edit.** Configure the source-column mapping and analysis settings there, then run every cell from top to bottom.

Required inputs are:

1. an observed outcome coded as 0/1 or Y/N;
2. a model prediction score expressed as a 0-1 probability or 0-100 percentage; and
3. an encounter or prediction time expressed as YYYYMM or a parseable datetime.

Optional inputs include race/ethnicity, sex or gender, HPI, numeric age or age group, patient or encounter identifier, and a cohort field. Missing optional mapped columns are ignored. Choose a clinically justified classification threshold for converting model scores to positive or negative predictions. Review the repository README for setup, privacy, and interpretation guidance.

## Configuration (Edit This Cell Only)

Update the settings and column mapping in the code cell below. No other notebook cells need to be edited; after configuration, run the notebook from top to bottom.

In [ ]:
# This is the only code cell users need to edit.
# After configuring it, run the notebook from top to bottom.
# Inputs
data_dir = "Please edit this variable for data's directory (e.g., './data/')"
data_name = "Please edit this variable for data's filename (e.g., 'model_predictions.csv')"
model_name = (
    "Please edit this variable for model name and version (e.g., 'CKD-HF-Cancer Model v2.0')"
)
project_info = "Please edit this variable for project information (e.g., 'CKD-HF-Cancer Model v2.0, trained on 2023-01-01 to 2023-06-30 data, evaluated on 2023-07-01 to 2023-12-31 data')"
id_unit = "patient"  # Required when id_var is mapped: choose "patient" or "encounter".

# Runtime settings
n_bootstrap = 1000  # NOTE: 50 is suitable for development/testing only.
# Increase to >= 500 for research or production use.
n_permutation = 1000  # NOTE: same -- increase to >= 1000 for stable results.
MIN_GROUP_N = 100  # Shared minimum usable rows for every attribute-specific analysis.
random_seed = 42  # Single seed used by all stochastic functions for reproducibility.
classification_threshold = 0.2  # Clinically justified score cutoff for positive predictions.

# Task 3 disparity threshold
disparity_ratio = 0.2  # The threshold for disparity ratio used in the analysis.
ratio_low = 1 - disparity_ratio
ratio_high = 1 + disparity_ratio

# Age binning (optional): edit to customize the age groups; leave as-is for the default.
age_bins = [0, 18, 65, float("inf")]
age_labels = ["0-18 years", "18-65 years", "65+ years"]

# Column mapping: source column -> template variable
column_mapping = {
    "patientdurablekey": "id_var",
    "encmonth": "time_var",
    "y_ObservedOutcome": "final_outcome",
    "y_prob": "probability",
    "age_group_str": "age_group",
    "sex": "gender_identity",
    "DerivedRaceEthnicity": "race_ethnicity",
    "CAHPIQuartile": "hpi_score",
    "cohort_flag": "cohort_flag_with_copd_asthma",
}
column_mapping = {
    str(src).strip(): str(dst).strip()
    for src, dst in column_mapping.items()
    if str(src).strip() and str(dst).strip()
}
# Cohort subsetting (optional): filter rows by one or more columns before analysis.
# Use a dict where each key is a column name and the value is a list of values to keep.
# All filters are applied with AND logic (rows must satisfy every filter).
# Set subset_filters = {} to skip subsetting (use all rows).
#
# Examples:
#   subset_filters = {"cohort_flag_with_copd_asthma": ["Cancer"]}        → keep only Cancer
#   subset_filters = {"cohort_flag_with_copd_asthma": ["Cancer", "HF"]}  → keep Cancer or HF
#   subset_filters = {                                                    → Cancer AND Female
#       "cohort_flag_with_copd_asthma": ["Cancer"],
#       "gender_identity": ["Female"],
#   }
#   subset_filters = {}                                                   → no subsetting
subset_filters = {}

# Figure and table numbering
iteration_fig = 0
iteration_tbl = 0

In [ ]:
# Install the project dependencies once before running this notebook:
# %pip install -r requirements.txt

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint
from statsmodels.tools.sm_exceptions import (
    DomainWarning,
    PerfectSeparationError,
    PerfectSeparationWarning,
)
from scipy import stats
from scipy.special import logit
from scipy.stats import norm
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from IPython.display import HTML, Markdown, display

warnings.simplefilter("ignore", DomainWarning)
pd.options.display.float_format = "{:.6f}".format

In [ ]:
file_path = Path(data_dir) / data_name
if not file_path.exists():
    raise FileNotFoundError(f"Input file not found: {file_path}")

ext = file_path.suffix.lower().lstrip(".")
na_tokens = ["NULL", "null", ""]

# Read
if ext in ["xlsx", "xls"]:
    df = pd.read_excel(file_path, na_values=na_tokens)
elif ext in ["parquet", "pq"]:
    df = pd.read_parquet(file_path)
else:
    df = pd.read_csv(file_path, na_values=na_tokens, keep_default_na=True, dtype={0: "string"})

# Rename
df = df.rename(columns=column_mapping)

# Keep mapped columns that are actually present. Demographic and cohort columns are optional.
mapped_cols = list(dict.fromkeys(column_mapping.values()))
required_cols = {"time_var", "final_outcome", "probability"}
missing_required = sorted(required_cols - set(df.columns))
if missing_required:
    raise ValueError(
        "Missing required mapped column(s): "
        + ", ".join(missing_required)
        + ". Check column_mapping in the configuration cell."
    )
available_mapped_cols = [col for col in mapped_cols if col in df.columns]
df = df[available_mapped_cols].copy()

has_race_ethnicity = "race_ethnicity" in df.columns
has_gender_identity = "gender_identity" in df.columns
has_age_group = "age_group" in df.columns
has_hpi_score = "hpi_score" in df.columns

# Normalize blanks to NaN
df = df.replace(r"^\s*$", np.nan, regex=True)

# Validate and normalize the observed outcome. Accepted encodings are 0/1 or Y/N.
outcome_raw = df["final_outcome"]
outcome_non_null = outcome_raw.dropna().astype(str).str.strip()
outcome_upper = outcome_non_null.str.upper()
if not outcome_non_null.empty and outcome_upper.isin(["Y", "N"]).all():
    df.loc[outcome_non_null.index, "final_outcome"] = outcome_upper.map({"Y": 1, "N": 0})
else:
    outcome_numeric = pd.to_numeric(outcome_raw, errors="coerce")
    invalid_outcome = outcome_raw.notna() & (outcome_numeric.isna() | ~outcome_numeric.isin([0, 1]))
    if invalid_outcome.any():
        invalid_values = outcome_raw.loc[invalid_outcome].astype(str).unique()[:5]
        raise ValueError(
            "final_outcome must contain only 0/1 or Y/N. "
            f"Invalid value(s) include: {invalid_values.tolist()}"
        )
    df["final_outcome"] = outcome_numeric
df["final_outcome"] = pd.to_numeric(df["final_outcome"], errors="coerce")

# Normalize/validate probability values before NA counting
df["probability"] = pd.to_numeric(df["probability"], errors="coerce")
prob_non_null = df["probability"].dropna()
if not prob_non_null.empty:
    prob_min = prob_non_null.min()
    prob_max = prob_non_null.max()
    if prob_min >= 0 and prob_max <= 1:
        pass
    elif prob_min >= 0 and prob_max <= 100:
        df["probability"] = df["probability"] / 100.0
df.loc[~df["probability"].between(0, 1), "probability"] = np.nan

# Count rows with NA in essential analysis columns only.
# Rows missing only a grouping attribute (e.g. HPI) are KEPT, so each subgroup analysis
# drops just its own missing values downstream instead of shrinking every analysis.
n_total = len(df)
essential_cols = ["final_outcome", "probability"]
rows_with_any_na = df[essential_cols].isna().any(axis=1)
n_dropped_na = int(rows_with_any_na.sum())
pct_dropped_na = (n_dropped_na / n_total * 100.0) if n_total else 0.0

# Drop them
df = df.loc[~rows_with_any_na].copy()
if df.empty:
    raise ValueError("No usable rows remain after validating outcome and probability values.")

# Continue pipeline. The ID unit controls labels and the interpretation of clustered inference.
if "id_var" not in df.columns:
    df["id_var"] = np.arange(1, len(df) + 1)
    id_unit_effective = "row"
else:
    id_unit_effective = str(globals().get("id_unit", "")).strip().lower()
    if id_unit_effective not in {"patient", "encounter"}:
        raise ValueError("id_unit must be 'patient' or 'encounter' when id_var is mapped.")

In [ ]:
# Clean and prepare the analysis data.
data = df.copy()

# Apply cohort subset filters if configured
if subset_filters:
    _n_before = len(data)
    _missing_filter_cols = sorted(set(subset_filters) - set(data.columns))
    if _missing_filter_cols:
        raise KeyError(
            "subset_filters contains unavailable column(s): " + ", ".join(_missing_filter_cols)
        )
    for _col, _vals in subset_filters.items():
        if _vals is not None:
            data = data[data[_col].isin(_vals)].copy()
    if data.empty:
        raise ValueError("No rows remain after applying subset_filters.")

# Auto-detect 6-digit YYYYMM integers (e.g. 202401) vs standard datetime strings
_time_raw = data["time_var"]
_time_num = pd.to_numeric(_time_raw, errors="coerce")
if _time_num.dropna().between(100000, 999999).all() and _time_num.notna().any():
    # Treat as YYYYMM integer
    data["year_month"] = (
        pd.to_datetime(_time_num.astype("Int64").astype(str), format="%Y%m", errors="coerce")
        .dt.to_period("M")
        .astype(str)
    )
else:
    data["year_month"] = (
        pd.to_datetime(data["time_var"], errors="coerce")
        .dt.tz_localize(None)
        .dt.to_period("M")
        .astype(str)
    )

if data["year_month"].eq("NaT").all():
    raise ValueError("time_var could not be parsed as YYYYMM values or datetimes.")

# Derive prediction from probability and threshold
data["prediction"] = (
    pd.to_numeric(data["probability"], errors="coerce") >= classification_threshold
).astype(int)

# Convert age into age group (bins/labels come from the Configuration cell; default if unset)
_age_bins = age_bins if "age_bins" in globals() else [0, 18, 65, float("inf")]
_age_labels = (
    age_labels if "age_labels" in globals() else ["0-18 years", "18-65 years", "65+ years"]
)
if has_age_group:
    # Bin a predominantly numeric age column; preserve any pre-binned text labels.
    _age_numeric = pd.to_numeric(data["age_group"], errors="coerce")
    _age_is_numeric = _age_numeric.notna().mean() > 0.9
    if _age_is_numeric:
        data["age_group"] = pd.cut(
            _age_numeric, bins=_age_bins, labels=_age_labels, right=False
        ).astype("category")
    else:
        data["age_group"] = data["age_group"].astype("category")


# HPI: accept a continuous score (quartiled here) OR a pre-binned quartile/category
# (auto-detected: numeric dtype OR >90% numeric-parseable → bin into quartiles;
#  otherwise treat as pre-binned categorical and cast to category)
def _qcut_with_available_labels(series, q=4, labels=("Q1", "Q2", "Q3", "Q4")):
    non_null = series.dropna()
    if non_null.nunique() <= 1:
        result = pd.Series(pd.NA, index=series.index, dtype="string")
        result.loc[series.notna()] = labels[0]
        return result.astype("category")
    binned = pd.qcut(series, q=q, duplicates="drop")
    n_bins = len(binned.cat.categories)
    return binned.cat.rename_categories(list(labels)[:n_bins])


def _bin_or_keep(series, q=4, labels=("Q1", "Q2", "Q3", "Q4")):
    # Already a numeric dtype → check if values are exactly {1,2,3,4} (pre-binned quartile integers)
    if pd.api.types.is_numeric_dtype(series):
        unique_vals = set(series.dropna().unique())
        if unique_vals <= {1, 2, 3, 4}:
            # Map integer quartile labels directly
            return series.map({1: "Q1", 2: "Q2", 3: "Q3", 4: "Q4"}).astype("category")
        return _qcut_with_available_labels(series, q=q, labels=labels)
    # String/object column: try to parse as numeric
    s_num = pd.to_numeric(series, errors="coerce")
    is_numeric = s_num.notna().mean() > 0.9
    if is_numeric:
        unique_vals = set(s_num.dropna().unique())
        if unique_vals <= {1, 2, 3, 4}:
            return s_num.map({1: "Q1", 2: "Q2", 3: "Q3", 4: "Q4"}).astype("category")
        return _qcut_with_available_labels(s_num, q=q, labels=labels)
    # Pre-binned categorical: keep as-is
    return series.astype("category")


if has_hpi_score:
    data["hpi_quartile"] = _bin_or_keep(data["hpi_score"])

core_cols = ["id_var", "time_var", "year_month", "final_outcome", "prediction", "probability"]
optional_cols = []
if has_race_ethnicity and "race_ethnicity" in data.columns:
    optional_cols.append("race_ethnicity")
if has_gender_identity and "gender_identity" in data.columns:
    optional_cols.append("gender_identity")
if has_age_group and "age_group" in data.columns:
    optional_cols.append("age_group")
if has_hpi_score and "hpi_quartile" in data.columns:
    optional_cols.append("hpi_quartile")
if "cohort_flag_with_copd_asthma" in data.columns:
    optional_cols.append("cohort_flag_with_copd_asthma")
data = data[core_cols + optional_cols]

if not has_race_ethnicity:
    data["race_ethnicity"] = pd.NA
if not has_gender_identity:
    data["gender_identity"] = pd.NA
if not has_age_group:
    data["age_group"] = pd.NA
if not has_hpi_score:
    data["hpi_quartile"] = pd.NA

data["TP"] = [
    (1 if pred == 1 and actual == 1 else 0)
    for pred, actual in zip(data["prediction"], data["final_outcome"])
]
data["TN"] = [
    (1 if pred == 0 and actual == 0 else 0)
    for pred, actual in zip(data["prediction"], data["final_outcome"])
]
data["FP"] = [
    (1 if pred == 1 and actual == 0 else 0)
    for pred, actual in zip(data["prediction"], data["final_outcome"])
]
data["FN"] = [
    (1 if pred == 0 and actual == 1 else 0)
    for pred, actual in zip(data["prediction"], data["final_outcome"])
]

# Clip logit scores to ±4.5951, which corresponds to probabilities in [0.01, 0.99].
# This prevents ±inf from entering the calibration model and is equivalent to
# clamping extreme model scores. Adjust via the _logit() helper's eps if needed.
data["Logit_p"] = logit(data["probability"])
data["Logit_p"] = data["Logit_p"].apply(lambda x: -4.5951 if x < -4.5951 or x == -np.inf else x)
data["Logit_p"] = data["Logit_p"].apply(lambda x: 4.5951 if x > 4.5951 or x == np.inf else x)

## Performance Metric Definitions

In [ ]:
tbl_definition = pd.DataFrame(
    {
        "Metric": [
            "AUROC",
            "AUPRC",
            "Sensitivity",
            "Specificity",
            "PPV",
            "NPV",
            "Accuracy",
            "FDR",
            "FOR",
            "FPR",
            "FNR",
        ],
        "FullName": [
            "Area Under the ROC Curve",
            "Area Under the Precision-Recall Curve",
            "True Positive Rate",
            "True Negative Rate",
            "Positive Predictive Value",
            "Negative Predictive Value",
            "Classification Accuracy",
            "False Discovery Rate",
            "False Omission Rate",
            "False Positive Rate",
            "False Negative Rate",
        ],
        "Description": [
            "Area under the roc curve",
            "Area under the precision-recall curve",
            "True positives / Known positives",
            "True negatives / Known negatives",
            "True positives / Predicted positives",
            "True negatives / Predicted negatives",
            "(True positives + True negatives) / N",
            "False positives / Predicted positives",
            "False negatives / Predicted negatives",
            "False positives / Known negatives",
            "False negatives / Known positives",
        ],
    }
)

display(HTML(tbl_definition.to_html(index=False)))
iteration_tbl += 1
display(Markdown(f"**Table {iteration_tbl}: Summary of performance metric definitions.**"))

In [ ]:
display(
    Markdown(
        f"## {model_name}\n\n" f"This report evaluates the **{model_name}**. " f"{project_info}"
    )
)

In [ ]:
def _source_for(template_var):
    for src, dst in column_mapping.items():
        if dst == template_var:
            return src
    return "(not provided)"


variable_usage = pd.DataFrame(
    [
        {
            "Variable": "id_var",
            "Source": _source_for("id_var"),
            "Used for outcomes/metrics": f"{id_unit_effective.title()}-level identifier used for counts and clustered inference.",
        },
        {
            "Variable": "time_var",
            "Source": _source_for("time_var"),
            "Used for outcomes/metrics": "Time indexing and month-based trend summaries.",
        },
        {
            "Variable": "final_outcome",
            "Source": _source_for("final_outcome"),
            "Used for outcomes/metrics": "Observed outcome label (ground truth) for all performance and fairness metrics.",
        },
        {
            "Variable": "prediction",
            "Source": "Derived from probability >= classification_threshold",
            "Used for outcomes/metrics": "Predicted class for TP/TN/FP/FN and threshold-based fairness performance measures.",
        },
        {
            "Variable": "probability",
            "Source": _source_for("probability"),
            "Used for outcomes/metrics": "Risk score used for AUROC, AUPRC, calibration, thresholding, and disparity analyses.",
        },
        {
            "Variable": "age_group",
            "Source": _source_for("age_group"),
            "Used for outcomes/metrics": "Subgroup fairness/performance outcomes by age category.",
        },
        {
            "Variable": "gender_identity",
            "Source": _source_for("gender_identity"),
            "Used for outcomes/metrics": "Subgroup fairness/performance outcomes by sex/gender group.",
        },
        {
            "Variable": "race_ethnicity",
            "Source": _source_for("race_ethnicity"),
            "Used for outcomes/metrics": "Subgroup fairness/performance outcomes by race/ethnicity group.",
        },
        {
            "Variable": "hpi_score -> hpi_quartile",
            "Source": _source_for("hpi_score"),
            "Used for outcomes/metrics": "Converted to HPI quartiles for subgroup fairness/performance outcomes.",
        },
        {
            "Variable": "TP / TN / FP / FN (derived)",
            "Source": "Derived from prediction + final_outcome",
            "Used for outcomes/metrics": "Confusion-matrix metrics (Sensitivity, Specificity, PPV, NPV, FPR, FNR, FDR, FOR, Accuracy).",
        },
    ]
)

display(HTML(variable_usage.to_html(index=False)))
iteration_tbl += 1
display(Markdown(f"**Table {iteration_tbl}: Variables and the outcomes/metrics they inform.**"))

## Task 1: Descriptive Statistics of Model Performance

For each attribute group, we summarize  

- Sample size;
- event prevalence at the patient, encounter, or row level, according to the configured identifier;
- confusion-matrix metrics at the selected classification threshold;
- area under the receiver operating characteristic (AUROC) curve;
- area under the precision-recall (AUPRC) curve; and
- model calibration.  

These summaries are provided for context and reference.


In [ ]:
display(
    Markdown(
        f"A classification threshold of {classification_threshold} was used to convert predicted probabilities "
        f"into binary predictions (positive vs. negative).\n\n"
        f"**Excluded {n_dropped_na} of {n_total} rows "
        f"({pct_dropped_na:.2f}%)** due to missing or invalid outcome/probability values, "
        f"leaving **{len(df)}** rows for analysis."
    )
)

### Sample Size, Prevalence, Confusion Matrix Entries

#### Race/Ethnicity (Confusion Matrix)

In [ ]:
def eligible_group_levels(
    df, group_col, outcome_col="final_outcome", score_col="probability", min_n=MIN_GROUP_N
):
    """Return eligible levels for one attribute without changing the master dataset."""
    check = df[[group_col, outcome_col, score_col]].copy()
    check[outcome_col] = pd.to_numeric(check[outcome_col], errors="coerce")
    check[score_col] = pd.to_numeric(check[score_col], errors="coerce")
    check = check.dropna(subset=[group_col, outcome_col, score_col])
    check = check.loc[check[outcome_col].isin([0, 1])].copy()
    check[group_col] = check[group_col].astype(str)
    eligibility = check.groupby(group_col, observed=True)[outcome_col].agg(
        N="size", OutcomeClasses="nunique"
    )
    keep = eligibility.index[(eligibility["N"] >= min_n) & (eligibility["OutcomeClasses"] == 2)]
    return keep.astype(str).tolist()


def filter_group_analysis_df(
    df, group_col, outcome_col="final_outcome", score_col="probability", min_n=MIN_GROUP_N
):
    """Filter a copy for one attribute analysis; never filter `data` globally."""
    keep = eligible_group_levels(
        df, group_col, outcome_col=outcome_col, score_col=score_col, min_n=min_n
    )
    mask = df[group_col].notna() & df[group_col].astype(str).isin(keep)
    return df.loc[mask].copy()


def build_summary(df, group_col, group_rename, id_col="id_var", min_n=MIN_GROUP_N):
    df_valid = filter_group_analysis_df(df, group_col, min_n=min_n)
    unit = str(globals().get("id_unit_effective", "row")).strip().lower()
    unit_title = {"patient": "Patient", "encounter": "Encounter", "row": "Row"}.get(unit, "ID")
    count_col = f"# Unique {unit_title}s"
    prevalence_col = f"{unit_title}-Level Prevalence"
    output_cols = [
        group_rename,
        count_col,
        prevalence_col,
        "Encounter Event Rate",
        "Total # Scores",
        "TP",
        "TN",
        "FP",
        "FN",
    ]
    if df_valid.empty:
        return pd.DataFrame(columns=output_cols)
    out = (
        df_valid.groupby(group_col, observed=True)
        .apply(
            lambda group: pd.Series(
                {
                    count_col: group[id_col].nunique(dropna=True),
                    prevalence_col: (
                        group.groupby(id_col)["final_outcome"].max().mean()
                        if len(group)
                        else np.nan
                    ),
                    "Encounter Event Rate": group["final_outcome"].mean() if len(group) else np.nan,
                    "Total # Scores": len(group),
                    "TP": ((group["prediction"] == 1) & (group["final_outcome"] == 1)).sum(),
                    "TN": ((group["prediction"] == 0) & (group["final_outcome"] == 0)).sum(),
                    "FP": ((group["prediction"] == 1) & (group["final_outcome"] == 0)).sum(),
                    "FN": ((group["prediction"] == 0) & (group["final_outcome"] == 1)).sum(),
                }
            ),
            include_groups=False,
        )
        .reset_index()
        .rename(columns={group_col: group_rename})
    )

    int_cols = [count_col, "Total # Scores", "TP", "TN", "FP", "FN"]
    out[int_cols] = out[int_cols].apply(pd.to_numeric, errors="coerce").astype("Int64")
    out[prevalence_col] = pd.to_numeric(out[prevalence_col], errors="coerce")
    out["Encounter Event Rate"] = pd.to_numeric(out["Encounter Event Rate"], errors="coerce")

    return out


def format_summary_table_for_display(tbl: pd.DataFrame) -> pd.DataFrame:
    display_tbl = tbl.copy()
    int_cols = [c for c in display_tbl.columns if c.startswith("# Unique ")] + [
        c for c in ["Total # Scores", "TP", "TN", "FP", "FN"] if c in display_tbl.columns
    ]
    for col in int_cols:
        display_tbl[col] = display_tbl[col].map(lambda x: "" if pd.isna(x) else f"{int(x)}")
    rate_cols = [c for c in display_tbl.columns if c.endswith("-Level Prevalence")] + [
        "Encounter Event Rate"
    ]
    for rate_col in rate_cols:
        if rate_col in display_tbl.columns:
            display_tbl[rate_col] = pd.to_numeric(display_tbl[rate_col], errors="coerce").map(
                lambda x: "" if pd.isna(x) else f"{float(x):.3f}"
            )
    return display_tbl


if has_race_ethnicity:
    display(
        HTML(
            format_summary_table_for_display(
                build_summary(data, "race_ethnicity", "Race Ethnicity")
            ).to_html(index=False)
        )
    )
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Sample size, prevalence and confusion matrix counts by race/ethnicity.**"
        )
    )
else:
    display(
        Markdown(
            "**Race/ethnicity data not available, skipping related summary and performance panels.**"
        )
    )

In [ ]:
# ============================================================
# Configuration (what to analyze / how to plot)
# ============================================================
Dataset = data
d = "race_ethnicity"
Observed_Outcome = "final_outcome"
Score = "probability"
DatasetName = "main"

MIN_N = MIN_GROUP_N  # shared minimum for tables, plots, estimates, tests, and Task 3
# Eligibility is calculated separately for each attribute.
CALIB_BINS = 10  # number of bins for calibration curve
thresholds = np.round(np.arange(0.0, 1.01, 0.1), 2)

# Metrics to plot as a function of threshold
MetricsList = ["Sensitivity", "Specificity", "FPR", "FNR", "PPV", "NPV", "FOR", "FDR"]


# ============================================================
# Data preparation helpers
# ============================================================
def _prep_df(Dataset, d, Observed_Outcome, Score):
    """Subset to needed columns, coerce types, drop invalid rows, sort by group."""
    Temp = Dataset[[d, Observed_Outcome, Score]].copy()
    # Filter out NaN in group column before processing
    Temp = Temp[Temp[d].notna()].copy()
    # Convert to string but preserve NaN (astype(str) turns NaN into "nan").
    # Cast to object first to strip any Categorical dtype before the where().
    _col = Temp[d]
    Temp[d] = _col.astype(object).where(_col.isna(), _col.astype(str))
    Temp[Observed_Outcome] = pd.to_numeric(Temp[Observed_Outcome], errors="coerce")
    Temp[Score] = pd.to_numeric(Temp[Score], errors="coerce")
    Temp = Temp.dropna(subset=[d, Observed_Outcome, Score])
    Temp = Temp[Temp[Observed_Outcome].isin([0, 1])].copy()
    Temp = Temp.sort_values(by=d)
    Temp[Observed_Outcome] = Temp[Observed_Outcome].astype(int)
    return Temp


def _eligible_groups(Temp, d, Observed_Outcome, min_n):
    """Keep groups with enough sample size and both outcome classes present."""
    ok = []
    for grp, g in Temp.groupby(d, observed=True):
        if len(g) >= min_n and g[Observed_Outcome].nunique() == 2:
            ok.append(grp)
    return ok


# ============================================================
# Metric math helpers (confusion-matrix-derived metrics)
# ============================================================
def _compute_conf_metrics(tp, tn, fp, fn):
    """Compute common binary classification metrics from TP/TN/FP/FN counts."""
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
    ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
    forr = fn / (fn + tn) if (fn + tn) > 0 else np.nan
    fdr = fp / (fp + tp) if (fp + tp) > 0 else np.nan
    return sens, spec, fpr, fnr, ppv, npv, forr, fdr


# ============================================================
# Plot styling helper
# ============================================================
def apply_common_style(ax):
    """Apply consistent styling across panels."""
    ax.grid(True)


# ============================================================
# Panel plots: AUROC / Calibration / PR
# ============================================================
def _plot_roc(ax, Temp, groups, d, ycol, scol, show_overall=True):
    if show_overall:
        y = Temp[ycol].to_numpy()
        s = Temp[scol].to_numpy()
        fpr, tpr, _ = roc_curve(y, s)
        auc_overall = roc_auc_score(y, s)
        ax.plot(fpr, tpr, label=f"Overall (AUC={auc_overall:.2f}, N={len(Temp)})")

    for grp in groups:
        g = Temp[Temp[d] == grp]
        yg = g[ycol].to_numpy()
        sg = g[scol].to_numpy()
        fpr_g, tpr_g, _ = roc_curve(yg, sg)
        auc_g = roc_auc_score(yg, sg)
        ax.plot(fpr_g, tpr_g, label=f"{grp} (AUC={auc_g:.2f}, N={len(g)})")

    ax.plot([0, 1], [0, 1], linestyle="--", color="red", label="Random")
    ax.plot([0, 0, 1], [0, 1, 1], linestyle=":", color="green", label="Perfect")
    ax.set_xlim([-0.05, 1.05])
    ax.set_ylim([-0.05, 1.05])
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("AUROC")
    apply_common_style(ax)


def _plot_calibration(ax, Temp, groups, d, ycol, scol, n_bins, show_overall=True):
    if show_overall:
        y = Temp[ycol].to_numpy()
        s = Temp[scol].to_numpy()
        prob_true, prob_pred = calibration_curve(y, s, n_bins=n_bins)
        brier = brier_score_loss(y, s)
        ax.plot(prob_pred, prob_true, "o-", label=f"Overall (Brier={brier:.3f}, N={len(Temp)})")

    for grp in groups:
        g = Temp[Temp[d] == grp]
        yg = g[ycol].to_numpy()
        sg = g[scol].to_numpy()
        pt_g, pp_g = calibration_curve(yg, sg, n_bins=n_bins)
        brier_g = brier_score_loss(yg, sg)
        ax.plot(pp_g, pt_g, "o-", label=f"{grp} (Brier={brier_g:.3f}, N={len(g)})")

    ax.plot([0, 1], [0, 1], linestyle="--", color="black", label="Ideal")
    ax.set_xlim([-0.05, 1.05])
    ax.set_ylim([-0.05, 1.05])
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction positive")
    ax.set_title("Calibration")
    apply_common_style(ax)


def _plot_pr(ax, Temp, groups, d, ycol, scol, show_overall=True):
    if show_overall:
        y = Temp[ycol].to_numpy()
        s = Temp[scol].to_numpy()
        precision, recall, _ = precision_recall_curve(y, s)
        ap_overall = average_precision_score(y, s)
        ax.plot(recall, precision, label=f"Overall (AP={ap_overall:.2f}, N={len(Temp)})")

    for grp in groups:
        g = Temp[Temp[d] == grp]
        yg = g[ycol].to_numpy()
        sg = g[scol].to_numpy()
        p_g, r_g, _ = precision_recall_curve(yg, sg)
        ap_g = average_precision_score(yg, sg)
        ax.plot(r_g, p_g, label=f"{grp} (AP={ap_g:.2f}, N={len(g)})")

    ax.plot([1, 1, 0], [0, 1, 1], linestyle=":", color="green", label="Perfect")
    ax.set_xlim([-0.05, 1.05])
    ax.set_ylim([-0.05, 1.05])
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision–Recall")
    apply_common_style(ax)


# ============================================================
# Panel plot: metric value vs threshold (per group)
# ============================================================
def _plot_metric_vs_threshold(ax, Temp, groups, d, ycol, scol, metric_name, thresholds):
    """Plot metric across decision thresholds for each group."""
    for grp in groups:
        g = Temp[Temp[d] == grp]
        y = g[ycol].to_numpy()
        s = g[scol].to_numpy()

        xs, ys = [], []
        for thr in thresholds:
            p = (s >= thr).astype(int)
            tp = int(((p == 1) & (y == 1)).sum())
            tn = int(((p == 0) & (y == 0)).sum())
            fp = int(((p == 1) & (y == 0)).sum())
            fn = int(((p == 0) & (y == 1)).sum())
            sens, spec, fpr, fnr, ppv, npv, forr, fdr = _compute_conf_metrics(tp, tn, fp, fn)

            val = {
                "Sensitivity": sens,
                "Specificity": spec,
                "FPR": fpr,
                "FNR": fnr,
                "PPV": ppv,
                "NPV": npv,
                "FOR": forr,
                "FDR": fdr,
            }[metric_name]

            xs.append(thr)
            ys.append(val)

        ax.plot(xs, ys, marker="o", label=grp)

    ax.set_title(f"{metric_name}")
    ax.set_xlabel("Threshold")
    ax.set_ylabel(metric_name)
    ax.set_xticks(thresholds)
    plt.setp(ax.get_xticklabels(), rotation=60, ha="right")
    apply_common_style(ax)


def plot_group_performance_panels(
    Dataset,
    d,
    Observed_Outcome,
    Score,
    DatasetName="main",
    MIN_N=MIN_GROUP_N,
    CALIB_BINS=10,
    thresholds=None,
    MetricsList=None,
    figsize_per_row=4.5,
    ncols=2,
    fig_num=None,
    show=True,
    show_overall=False,
):
    if thresholds is None:
        thresholds = np.round(np.arange(0.0, 1.01, 0.1), 2)
    if MetricsList is None:
        MetricsList = ["Sensitivity", "Specificity", "FPR", "FNR", "PPV", "NPV", "FOR", "FDR"]

    Temp = _prep_df(Dataset, d, Observed_Outcome, Score)
    groups = eligible_group_levels(
        Dataset, d, outcome_col=Observed_Outcome, score_col=Score, min_n=MIN_N
    )
    Temp = Temp.loc[Temp[d].isin(groups)].copy()

    K = len(MetricsList) + 4
    nrows = int(np.ceil(K / ncols))

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(16, figsize_per_row * nrows),
        constrained_layout=True,
        num=fig_num,
    )
    axes = np.array(axes).reshape(-1)

    for i, m in enumerate(MetricsList):
        _plot_metric_vs_threshold(axes[i], Temp, groups, d, Observed_Outcome, Score, m, thresholds)

    start = len(MetricsList)
    _plot_roc(axes[start], Temp, groups, d, Observed_Outcome, Score, show_overall=show_overall)
    _plot_calibration(
        axes[start + 1],
        Temp,
        groups,
        d,
        Observed_Outcome,
        Score,
        CALIB_BINS,
        show_overall=show_overall,
    )
    _plot_pr(axes[start + 2], Temp, groups, d, Observed_Outcome, Score, show_overall=show_overall)

    legend_ax = axes[start + 3]
    legend_ax.axis("off")

    for j in range(start + 4, len(axes)):
        axes[j].axis("off")

    handles, labels = axes[0].get_legend_handles_labels()
    legend_ax.legend(
        handles,
        labels,
        loc="center",
        fontsize=12,
        frameon=False,
        ncol=1,
        handlelength=2.5,
        labelspacing=0.8,
        borderpad=1.0,
    )

    if show:
        plt.show()

In [ ]:
if has_race_ethnicity:
    plot_group_performance_panels(
        Dataset=data,
        d="race_ethnicity",
        Observed_Outcome="final_outcome",
        Score="probability",
        DatasetName="main",
        MIN_N=MIN_GROUP_N,
        CALIB_BINS=10,
        thresholds=np.round(np.arange(0.0, 1.01, 0.1), 2),
        MetricsList=["Sensitivity", "Specificity", "FPR", "FNR", "PPV", "NPV", "FOR", "FDR"],
    )
    iteration_fig += 1
    display(Markdown(f"**Figure {iteration_fig}: Model performance panels by race/ethnicity.**"))

#### Sex (Confusion Matrix)

In [ ]:
if has_gender_identity:
    display(
        HTML(
            format_summary_table_for_display(build_summary(data, "gender_identity", "Sex")).to_html(
                index=False
            )
        )
    )
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Sample size, prevalence and confusion matrix counts by sex.**"
        )
    )
else:
    display(
        Markdown(
            "**Sex/gender data not available, skipping related summary and performance panels.**"
        )
    )

In [ ]:
if has_gender_identity:
    plot_group_performance_panels(
        Dataset=data,
        d="gender_identity",
        Observed_Outcome="final_outcome",
        Score="probability",
        DatasetName="main",
        MIN_N=MIN_GROUP_N,
        CALIB_BINS=10,
        thresholds=np.round(np.arange(0.0, 1.01, 0.1), 2),
        MetricsList=["Sensitivity", "Specificity", "FPR", "FNR", "PPV", "NPV", "FOR", "FDR"],
    )
    iteration_fig += 1
    display(Markdown(f"**Figure {iteration_fig}: Model performance panels by sex.**"))

#### Age (Confusion Matrix)

In [ ]:
if has_age_group:
    display(
        HTML(
            format_summary_table_for_display(build_summary(data, "age_group", "Age")).to_html(
                index=False
            )
        )
    )
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Sample size, prevalence and confusion matrix counts by age.**"
        )
    )
else:
    display(
        Markdown(
            "**Age group data not available, skipping related summary and performance panels.**"
        )
    )

In [ ]:
if has_age_group:
    plot_group_performance_panels(
        Dataset=data,
        d="age_group",
        Observed_Outcome="final_outcome",
        Score="probability",
        DatasetName="main",
        MIN_N=MIN_GROUP_N,
        CALIB_BINS=10,
        thresholds=np.round(np.arange(0.0, 1.01, 0.1), 2),
        MetricsList=["Sensitivity", "Specificity", "FPR", "FNR", "PPV", "NPV", "FOR", "FDR"],
    )
    iteration_fig += 1
    display(Markdown(f"**Figure {iteration_fig}: Model performance panels by age group.**"))

#### HPI (Confusion Matrix)

In [ ]:
if has_hpi_score:
    display(
        HTML(
            format_summary_table_for_display(
                build_summary(data, "hpi_quartile", "HPI Quartile")
            ).to_html(index=False)
        )
    )
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Sample size, prevalence and confusion matrix counts by HPI quartile.**"
        )
    )
else:
    display(
        Markdown(
            "**HPI score data not available, skipping related summary and performance panels.**"
        )
    )

In [ ]:
if has_hpi_score:
    plot_group_performance_panels(
        Dataset=data,
        d="hpi_quartile",
        Observed_Outcome="final_outcome",
        Score="probability",
        DatasetName="main",
        MIN_N=MIN_GROUP_N,
        CALIB_BINS=10,
        thresholds=np.round(np.arange(0.0, 1.01, 0.1), 2),
        MetricsList=["Sensitivity", "Specificity", "FPR", "FNR", "PPV", "NPV", "FOR", "FDR"],
    )
    iteration_fig += 1
    display(
        Markdown(f"**Figure {iteration_fig}: Model performance panels by hpi group (quartiles).**")
    )

### Point Estimates and Confidence Limits 

Estimates and 95% confidence limits are shown for all metrics by group tabularly and graphically. 

In [ ]:
# ----------------------------
# Basic helpers
# ----------------------------
def has_multiple_clusters(df, id_col):
    return (
        (id_col in df.columns)
        and df[id_col].notna().any()
        and (df[id_col].nunique(dropna=True) > 1)
    )


def to_binary01(s):
    s = pd.Series(s)
    num = pd.to_numeric(s, errors="coerce")
    if num.notna().mean() > 0.9:
        return num.where(num.isin([0, 1]), np.nan)
    xs = s.astype(str).str.strip().str.lower()
    mapping = {
        "1": 1,
        "0": 0,
        "y": 1,
        "n": 0,
        "yes": 1,
        "no": 0,
        "true": 1,
        "false": 0,
        "t": 1,
        "f": 0,
    }
    return xs.map(mapping).astype("float")


def fmt_prob_ci(p, lcl, ucl):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return None
    if (
        lcl is None
        or ucl is None
        or (isinstance(lcl, float) and np.isnan(lcl))
        or (isinstance(ucl, float) and np.isnan(ucl))
    ):
        return f"{p:.6g} [NA, NA]"
    return f"{p:.6g} [{lcl:.6g}, {ucl:.6g}]"


def _logit_to_prob(x):
    return 1 / (1 + np.exp(-x))


def fit_binomial_glm_with_cluster(dd, formula, id_col="id_var"):
    model = sm.GLM.from_formula(formula, data=dd, family=sm.families.Binomial())
    # Statsmodels may emit PerfectSeparationWarning and still return unidentified estimates.
    # Convert it to an exception so callers can mark that inference unavailable.
    with warnings.catch_warnings():
        warnings.simplefilter("error", PerfectSeparationWarning)
        if has_multiple_clusters(dd, id_col):
            return model.fit(cov_type="cluster", cov_kwds={"groups": dd[id_col].astype(str).values})
        return model.fit()


def _concat_non_none(dfs):
    dfs = [x for x in dfs if isinstance(x, pd.DataFrame) and len(x) > 0]
    if not dfs:
        return None
    return pd.concat(dfs, ignore_index=True)


# ----------------------------
# Multiple testing adjustment
# ----------------------------
def _bh_adjust(p):
    p = np.asarray(p, dtype=float)
    out = np.full_like(p, np.nan, dtype=float)
    m = np.isfinite(p)
    if m.sum() == 0:
        return out
    pv = p[m]
    n = pv.size
    order = np.argsort(pv)
    ranked = pv[order]
    adj = ranked * n / (np.arange(1, n + 1))
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0, 1)
    out[m] = adj[np.argsort(order)]
    return out


def _holm_adjust(p):
    p = np.asarray(p, dtype=float)
    out = np.full_like(p, np.nan, dtype=float)
    m = np.isfinite(p)
    if m.sum() == 0:
        return out
    pv = p[m]
    n = pv.size
    order = np.argsort(pv)
    ranked = pv[order]
    holm = (n - np.arange(n)) * ranked
    holm = np.maximum.accumulate(holm)
    holm = np.clip(holm, 0, 1)
    back = np.empty_like(holm)
    back[order] = holm
    out[m] = back
    return out


# ----------------------------
# Cluster-robust CI for binomial rates
# ----------------------------
def binomial_rate_ci_cluster_robust(y01, clusters):
    y = pd.Series(y01).reset_index(drop=True)
    g = pd.Series(clusters).reset_index(drop=True)

    m = y.notna() & g.notna()
    y = y[m].astype(int)
    g = g[m].astype(str)

    n = len(y)
    if n == 0:
        return (None, None, None)

    p_hat = float(y.mean())
    if y.nunique() == 1 or g.nunique() < 2:
        return (p_hat, None, None)

    X = np.ones((n, 1))
    res = sm.GLM(y.values, X, family=sm.families.Binomial()).fit(
        cov_type="cluster", cov_kwds={"groups": g.values}
    )

    beta = float(res.params[0])
    se = float(res.bse[0])
    z = 1.96

    p = float(_logit_to_prob(beta))
    lcl = float(_logit_to_prob(beta - z * se))
    ucl = float(_logit_to_prob(beta + z * se))
    return (p, lcl, ucl)


# ----------------------------
# Per-group binomial metrics (NO AUC/AP bootstrap here)
# ----------------------------
def compute_binomial_metrics_ci(g, id_col="id_var"):
    y = g["y"].astype(int).to_numpy()
    pred = g["pred"].astype(int).to_numpy()
    cid = (
        g[id_col].astype(str).to_numpy()
        if id_col in g.columns
        else np.array([np.nan] * len(g), dtype=object)
    )

    ok = pd.notna(y) & pd.notna(pred) & pd.notna(cid)
    y = y[ok].astype(int)
    pred = pred[ok].astype(int)
    cid = cid[ok].astype(str)

    N = int(len(y))
    if N == 0:
        return {
            "N": 0,
            "Sensitivity": None,
            "Specificity": None,
            "PPV": None,
            "NPV": None,
            "Accuracy": None,
            "FPR": None,
            "FNR": None,
            "FDR": None,
            "FOR": None,
            "AUROC": None,
            "AUPRC": None,
        }

    sens = binomial_rate_ci_cluster_robust(pred[y == 1], cid[y == 1])
    spec = binomial_rate_ci_cluster_robust((1 - pred)[y == 0], cid[y == 0])
    ppv = binomial_rate_ci_cluster_robust(y[pred == 1], cid[pred == 1])
    npv = binomial_rate_ci_cluster_robust((1 - y)[pred == 0], cid[pred == 0])
    acc = binomial_rate_ci_cluster_robust((pred == y).astype(int), cid)
    fpr = binomial_rate_ci_cluster_robust(pred[y == 0], cid[y == 0])
    fnr = binomial_rate_ci_cluster_robust((1 - pred)[y == 1], cid[y == 1])
    fdr = binomial_rate_ci_cluster_robust((1 - y)[pred == 1], cid[pred == 1])
    for_ = binomial_rate_ci_cluster_robust(y[pred == 0], cid[pred == 0])

    return {
        "N": N,
        "Sensitivity": fmt_prob_ci(*sens),
        "Specificity": fmt_prob_ci(*spec),
        "PPV": fmt_prob_ci(*ppv),
        "NPV": fmt_prob_ci(*npv),
        "Accuracy": fmt_prob_ci(*acc),
        "FPR": fmt_prob_ci(*fpr),
        "FNR": fmt_prob_ci(*fnr),
        "FDR": fmt_prob_ci(*fdr),
        "FOR": fmt_prob_ci(*for_),
        "AUROC": None,
        "AUPRC": None,
    }


# ----------------------------
# Cluster bootstrap utilities
# ----------------------------
def build_cluster_index_map(cluster_ids):
    cluster_ids = np.asarray(cluster_ids).astype(str)
    m = {}
    for i, cid in enumerate(cluster_ids):
        m.setdefault(cid, []).append(i)
    for k in m:
        m[k] = np.asarray(m[k], dtype=int)
    return m


def _two_sided_p_from_diffs(diffs):
    diffs = np.asarray(diffs, dtype=float)
    diffs = diffs[np.isfinite(diffs)]
    if diffs.size == 0:
        return np.nan
    # Clamp each tail proportion to 0.5 before doubling to ensure p <= 1.0.
    # When all diffs == 0 both proportions equal 1.0, which would give p=2 without the clamp.
    return float(2.0 * min(np.mean(diffs > 0.0), np.mean(diffs < 0.0), 0.5))


# ----------------------------
# ONE JOINT BOOTSTRAP for AUROC/AUPRC:
#   - per-group CI strings
#   - pairwise p-values + Estimate = mean bootstrap (A - B)
#   - minimal output columns
# ----------------------------
def joint_cluster_bootstrap_auc_ap_minpairwise(
    prepped_df: pd.DataFrame,
    *,
    group_col: str,
    id_col: str = "id_var",
    score_col: str = "probability",
    B: int = 300,
    seed: int = 777,
    pairwise_adjust: str = "holm",
    ci_alpha: float = 0.05,
):
    d = prepped_df.dropna(subset=[group_col, "y", score_col, id_col]).copy()
    if d.empty or d["y"].nunique() != 2:
        return {"AUROC": {}, "AUPRC": {}}, None

    d[group_col] = d[group_col].astype(str)
    d[id_col] = d[id_col].astype(str)

    y = d["y"].astype(int).to_numpy()
    s = d[score_col].astype(float).to_numpy()
    g = d[group_col].to_numpy()
    cid = d[id_col].to_numpy()

    counts = pd.Series(g).value_counts()
    levels = counts.index.astype(str).tolist()
    if len(levels) < 1:
        return {"AUROC": {}, "AUPRC": {}}, None

    pairs = []
    for i, b in enumerate(levels[:-1]):
        for a in levels[i + 1 :]:
            pairs.append((a, b, int(counts.get(a, 0)), int(counts.get(b, 0))))

    cluster_index_map = build_cluster_index_map(cid)
    uniq = np.array(list(cluster_index_map.keys()), dtype=str)
    if uniq.size < 2:
        return {"AUROC": {}, "AUPRC": {}}, None

    rng = np.random.default_rng(seed)

    def stat_auc(yy, ss):
        return np.nan if np.unique(yy).size != 2 else float(roc_auc_score(yy, ss))

    def stat_ap(yy, ss):
        return np.nan if np.unique(yy).size != 2 else float(average_precision_score(yy, ss))

    # observed per-level for CI formatting
    obs_auc, obs_ap = {}, {}
    for lvl in levels:
        msk = g == lvl
        obs_auc[lvl] = stat_auc(y[msk], s[msk])
        obs_ap[lvl] = stat_ap(y[msk], s[msk])

    # boot per-level values
    boot_auc = {lvl: [] for lvl in levels}
    boot_ap = {lvl: [] for lvl in levels}

    for _ in range(B):
        samp = rng.choice(uniq, size=uniq.size, replace=True)
        idx = np.concatenate([cluster_index_map[k] for k in samp])

        yy = y[idx]
        ss = s[idx]
        gg = g[idx]

        for lvl in levels:
            msk = gg == lvl
            if msk.sum() == 0:
                boot_auc[lvl].append(np.nan)
                boot_ap[lvl].append(np.nan)
            else:
                boot_auc[lvl].append(stat_auc(yy[msk], ss[msk]))
                boot_ap[lvl].append(stat_ap(yy[msk], ss[msk]))

    lo_q = ci_alpha / 2.0
    hi_q = 1.0 - lo_q

    group_ci = {"AUROC": {}, "AUPRC": {}}
    for lvl in levels:
        pe = obs_auc.get(lvl, np.nan)
        arr = np.asarray(boot_auc[lvl], float)
        arr = arr[np.isfinite(arr)]
        if np.isfinite(pe) and arr.size >= max(30, int(0.2 * B)):
            lcl, ucl = np.quantile(arr, [lo_q, hi_q])
            group_ci["AUROC"][lvl] = fmt_prob_ci(float(pe), float(lcl), float(ucl))
        else:
            group_ci["AUROC"][lvl] = None

        pe = obs_ap.get(lvl, np.nan)
        arr = np.asarray(boot_ap[lvl], float)
        arr = arr[np.isfinite(arr)]
        if np.isfinite(pe) and arr.size >= max(30, int(0.2 * B)):
            lcl, ucl = np.quantile(arr, [lo_q, hi_q])
            group_ci["AUPRC"][lvl] = fmt_prob_ci(float(pe), float(lcl), float(ucl))
        else:
            group_ci["AUPRC"][lvl] = None

    if not pairs:
        return group_ci, None

    rows = []
    ref = "cluster_bootstrap"
    for metric, boot_map in (("AUROC", boot_auc), ("AUPRC", boot_ap)):
        for a, b, n_a, n_b in pairs:
            a_arr = np.asarray(boot_map[a], float)
            b_arr = np.asarray(boot_map[b], float)
            ok = np.isfinite(a_arr) & np.isfinite(b_arr)
            if metric == "AUROC":
                obs_a = obs_auc.get(a, np.nan)
                obs_b = obs_auc.get(b, np.nan)
            else:
                obs_a = obs_ap.get(a, np.nan)
                obs_b = obs_ap.get(b, np.nan)
            observed_diff = (
                float(obs_a - obs_b) if (np.isfinite(obs_a) and np.isfinite(obs_b)) else np.nan
            )
            # Use the observed difference (not the bootstrap mean) as the Estimate so that
            # AUROC/AUPRC rows are on the same scale as the binomial-metric rows.
            est_diff = observed_diff
            diffs = a_arr[ok] - b_arr[ok]
            p = _two_sided_p_from_diffs(diffs) if ok.sum() >= 1 else np.nan

            rows.append(
                {
                    "Metric": metric,
                    "Group_A": str(a),
                    "Group_B": str(b),
                    "Estimate": est_diff,
                    "Observed_Diff": observed_diff,
                    "p": p,
                    "N_A": int(n_a),
                    "N_B": int(n_b),
                    "reference": ref,
                    "p_adj": np.nan,
                    "Comparison": f"{a} -- {b}",
                }
            )

    out = pd.DataFrame(rows)

    # adjust within each metric
    adj = pairwise_adjust.lower()
    for m in out["Metric"].unique():
        mm = out["Metric"] == m
        if adj == "holm":
            out.loc[mm, "p_adj"] = _holm_adjust(out.loc[mm, "p"].to_numpy())
        elif adj == "bh":
            out.loc[mm, "p_adj"] = _bh_adjust(out.loc[mm, "p"].to_numpy())
        else:
            out.loc[mm, "p_adj"] = out.loc[mm, "p"].to_numpy()

    keep = [
        "Metric",
        "Group_A",
        "Group_B",
        "Estimate",
        "Observed_Diff",
        "p",
        "N_A",
        "N_B",
        "reference",
        "p_adj",
        "Comparison",
    ]
    out = (
        out[keep]
        .sort_values(by=["Metric", "N_B", "N_A"], ascending=[True, False, False])
        .reset_index(drop=True)
    )
    return group_ci, out


# ----------------------------
# Global p-values (unchanged from your approach)
# ----------------------------
def wald_test_group_difference_binomial(formula, data, id_col="id_var"):
    # Cast all categorical columns to str to avoid unused-level singularity issues
    data = data.copy()
    for col in data.select_dtypes(["category"]).columns:
        data[col] = data[col].astype(str)
    # Drop rows where the group variable is NaN/empty after cast
    group_col_match = re.search(r"C\(([^)]+)\)", formula)
    if group_col_match:
        gcol = group_col_match.group(1).strip()
        if gcol in data.columns:
            data = data[data[gcol].notna() & (data[gcol] != "nan")].copy()
    if len(data) == 0 or (group_col_match and data[gcol].nunique() < 2):
        return np.nan
    try:
        res = fit_binomial_glm_with_cluster(data, formula, id_col=id_col)
    except Exception:
        return np.nan
    names = res.model.data.param_names
    idx = [i for i, n in enumerate(names) if n.lower() != "intercept"]
    if not idx:
        return np.nan
    R = np.zeros((len(idx), len(names)))
    for r, j in enumerate(idx):
        R[r, j] = 1.0
    try:
        return float(res.wald_test(R, scalar=True).pvalue)
    except Exception:
        return np.nan


def group_auc_dispersion_statistic(y, s, g, metric_fn):
    overall = metric_fn(y, s)
    levels, counts = np.unique(g, return_counts=True)
    w = counts / counts.sum()
    vals = []
    for lvl in levels:
        m = g == lvl
        if np.unique(y[m]).size != 2:
            return np.nan
        vals.append(metric_fn(y[m], s[m]))
    vals = np.asarray(vals, float)
    return float(np.sum(w * (vals - overall) ** 2))


def permutation_test_auc_ap_by_group(
    d, group_col, id_col="id_var", score_col="probability", B=1000, seed=123
):
    d = d.dropna(subset=[group_col, "y", score_col, id_col]).copy()
    if len(d) == 0 or d["y"].nunique() != 2:
        return {"AUROC": np.nan, "AUPRC": np.nan}

    y = d["y"].astype(int).to_numpy()
    s = d[score_col].astype(float).to_numpy()
    g = d[group_col].astype(str).to_numpy()
    cid = d[id_col].astype(str).to_numpy()

    def auc_fn(yy, ss):
        return float(roc_auc_score(yy, ss))

    def ap_fn(yy, ss):
        return float(average_precision_score(yy, ss))

    obs_auc = group_auc_dispersion_statistic(y, s, g, auc_fn)
    obs_ap = group_auc_dispersion_statistic(y, s, g, ap_fn)

    tmp = pd.DataFrame({"cid": cid, "g": g})
    g_per_cluster = tmp.groupby("cid")["g"].agg(
        lambda x: x.mode().iloc[0] if len(x.mode()) else x.iloc[0]
    )
    cluster_ids = g_per_cluster.index.to_numpy()
    base_labels = g_per_cluster.to_numpy()

    cluster_to_rows = {}
    for i, c in enumerate(cid):
        cluster_to_rows.setdefault(c, []).append(i)
    for c in cluster_to_rows:
        cluster_to_rows[c] = np.asarray(cluster_to_rows[c], dtype=int)

    rng = np.random.default_rng(seed)
    ge_auc = n_auc = ge_ap = n_ap = 0
    for _ in range(B):
        perm_labels = rng.permutation(base_labels)
        gp = np.empty_like(g, dtype=object)
        for c, lab in zip(cluster_ids, perm_labels):
            gp[cluster_to_rows[c]] = lab

        st_auc = group_auc_dispersion_statistic(y, s, gp, auc_fn)
        st_ap = group_auc_dispersion_statistic(y, s, gp, ap_fn)

        if np.isfinite(st_auc) and np.isfinite(obs_auc):
            n_auc += 1
            ge_auc += st_auc >= obs_auc
        if np.isfinite(st_ap) and np.isfinite(obs_ap):
            n_ap += 1
            ge_ap += st_ap >= obs_ap

    p_auc = (ge_auc + 1) / (n_auc + 1) if n_auc else np.nan
    p_ap = (ge_ap + 1) / (n_ap + 1) if n_ap else np.nan
    return {"AUROC": p_auc, "AUPRC": p_ap}


def compute_group_difference_pvalues(
    prepped_df, group_col, id_col="id_var", score_col="probability", auc_perm_B=1000
):
    d = prepped_df.dropna(subset=[group_col, "y", "pred"]).copy()
    # Remove unused categorical levels so GLM sees only observed groups
    if hasattr(d[group_col], "cat"):
        d[group_col] = d[group_col].cat.remove_unused_categories()
    if len(d) == 0:
        return {}

    y = d["y"].astype(int)
    pred = d["pred"].astype(int)

    d["_correct"] = (pred == y).astype(int)
    d["_tn_ind"] = (1 - pred).astype(int)
    d["_fn_ind"] = (1 - pred).astype(int)
    d["_noty"] = (1 - y).astype(int)

    pv = {}
    dd = d.loc[d["y"] == 1]
    pv["Sensitivity"] = (
        wald_test_group_difference_binomial(f"pred ~ C({group_col})", dd, id_col=id_col)
        if len(dd)
        else np.nan
    )

    dd = d.loc[d["y"] == 0]
    pv["Specificity"] = (
        wald_test_group_difference_binomial(f"_tn_ind ~ C({group_col})", dd, id_col=id_col)
        if len(dd)
        else np.nan
    )
    pv["FPR"] = (
        wald_test_group_difference_binomial(f"pred ~ C({group_col})", dd, id_col=id_col)
        if len(dd)
        else np.nan
    )

    dd = d.loc[d["pred"] == 1]
    pv["PPV"] = (
        wald_test_group_difference_binomial(f"y ~ C({group_col})", dd, id_col=id_col)
        if len(dd)
        else np.nan
    )
    pv["FDR"] = (
        wald_test_group_difference_binomial(f"_noty ~ C({group_col})", dd, id_col=id_col)
        if len(dd)
        else np.nan
    )

    dd = d.loc[d["pred"] == 0]
    pv["NPV"] = (
        wald_test_group_difference_binomial(f"_noty ~ C({group_col})", dd, id_col=id_col)
        if len(dd)
        else np.nan
    )
    pv["FOR"] = (
        wald_test_group_difference_binomial(f"y ~ C({group_col})", dd, id_col=id_col)
        if len(dd)
        else np.nan
    )

    dd = d.loc[d["y"] == 1]
    pv["FNR"] = (
        wald_test_group_difference_binomial(f"_fn_ind ~ C({group_col})", dd, id_col=id_col)
        if len(dd)
        else np.nan
    )

    pv["Accuracy"] = wald_test_group_difference_binomial(
        f"_correct ~ C({group_col})", d, id_col=id_col
    )

    if score_col in d.columns and d[score_col].notna().any():
        pv.update(
            permutation_test_auc_ap_by_group(
                d, group_col, score_col=score_col, B=auc_perm_B, seed=random_seed
            )
        )
    else:
        pv["AUROC"] = np.nan
        pv["AUPRC"] = np.nan

    return pv


# ----------------------------
# Pairwise binomial metrics (minimal columns) + Estimate = risk difference (A - B)
# ----------------------------
def _identify_reference_level(res, group_col: str, levels_in_data: list[str]):
    names = list(res.model.data.param_names)
    prefix = f"C({group_col})[T."
    coef_levels = []
    for nm in names:
        if nm.startswith(prefix):
            coef_levels.append(nm.split("[T.", 1)[1].rstrip("]"))
    ref_candidates = [lvl for lvl in levels_in_data if lvl not in coef_levels]
    if len(ref_candidates) != 1:
        raise RuntimeError(
            f"Could not uniquely identify reference level for {group_col}. "
            f"Candidates={ref_candidates}, coef_levels={coef_levels}, levels={levels_in_data}"
        )
    return ref_candidates[0], coef_levels


def _prep_metric_subset(prepped_df: pd.DataFrame, metric_name: str, group_col: str):
    d = prepped_df.dropna(subset=[group_col, "y", "pred"]).copy()
    d[group_col] = d[group_col].astype(str)

    if metric_name == "Sensitivity":
        dd = d.loc[d["y"] == 1].copy()
        y_col = "pred"
    elif metric_name == "Specificity":
        dd = d.loc[d["y"] == 0].copy()
        dd["_tn_ind"] = (1 - dd["pred"]).astype(int)
        y_col = "_tn_ind"
    elif metric_name == "FPR":
        dd = d.loc[d["y"] == 0].copy()
        y_col = "pred"
    elif metric_name == "FNR":
        dd = d.loc[d["y"] == 1].copy()
        dd["_fn_ind"] = (1 - dd["pred"]).astype(int)
        y_col = "_fn_ind"
    elif metric_name == "PPV":
        dd = d.loc[d["pred"] == 1].copy()
        y_col = "y"
    elif metric_name == "NPV":
        dd = d.loc[d["pred"] == 0].copy()
        dd["_noty"] = (1 - dd["y"]).astype(int)
        y_col = "_noty"
    elif metric_name == "FDR":
        dd = d.loc[d["pred"] == 1].copy()
        dd["_noty"] = (1 - dd["y"]).astype(int)
        y_col = "_noty"
    elif metric_name == "FOR":
        dd = d.loc[d["pred"] == 0].copy()
        y_col = "y"
    elif metric_name == "Accuracy":
        dd = d.copy()
        dd["_correct"] = (dd["pred"] == dd["y"]).astype(int)
        y_col = "_correct"
    else:
        raise ValueError(f"Unknown metric_name: {metric_name}")

    dd = dd.dropna(subset=[y_col, group_col]).copy()
    return dd, y_col


def _ordered_pairs_by_frequency(dd: pd.DataFrame, group_col: str):
    counts = dd[group_col].value_counts()
    ordered_levels = counts.index.astype(str).tolist()
    pairs = []
    for i, b in enumerate(ordered_levels[:-1]):
        for a in ordered_levels[i + 1 :]:
            pairs.append((a, b, int(counts.get(a, 0)), int(counts.get(b, 0))))
    return pairs


def _pairwise_from_fitted_model_min(
    res, *, dd: pd.DataFrame, group_col: str, y_col: str, pairs, adjust: str
):
    levels_in_data = pd.Series(dd[group_col].values).unique().astype(str).tolist()
    names = list(res.model.data.param_names)
    V = res.cov_params()
    ref, coef_levels = _identify_reference_level(res, group_col, levels_in_data)

    level_to_param = {ref: None}
    for lvl in coef_levels:
        level_to_param[lvl] = f"C({group_col})[T.{lvl}]"

    # group means for risk-difference effect size (A - B)
    grp_mean = dd.groupby(group_col, observed=True)[y_col].mean()

    rows = []
    for a, b, n_a, n_b in pairs:
        c = np.zeros(len(names), dtype=float)
        pa = level_to_param.get(a, None)
        if pa is not None:
            c[names.index(pa)] += 1.0
        pb = level_to_param.get(b, None)
        if pb is not None:
            c[names.index(pb)] -= 1.0

        var = float(c @ V @ c)
        se = float(np.sqrt(var)) if var >= 0 else np.nan
        try:
            p = (
                float(res.wald_test(c, scalar=True).pvalue)
                if np.isfinite(se) and se > 0
                else np.nan
            )
        except Exception:
            p = np.nan

        est = float(
            grp_mean.get(str(a), np.nan) - grp_mean.get(str(b), np.nan)
        )  # observed RD (A - B)

        rows.append(
            {
                "Group_A": str(a),
                "Group_B": str(b),
                "Estimate": est,
                "Observed_Diff": est,
                "p": p,
                "N_A": int(n_a),
                "N_B": int(n_b),
                "reference": ref,
                "p_adj": np.nan,
                "Comparison": f"{a} -- {b}",
            }
        )

    out = pd.DataFrame(rows)

    adj = adjust.lower()
    if adj == "holm":
        out["p_adj"] = _holm_adjust(out["p"].values)
    elif adj == "bh":
        out["p_adj"] = _bh_adjust(out["p"].values)
    else:
        out["p_adj"] = out["p"].values

    keep = [
        "Group_A",
        "Group_B",
        "Estimate",
        "Observed_Diff",
        "p",
        "N_A",
        "N_B",
        "reference",
        "p_adj",
        "Comparison",
    ]
    out = out[keep].sort_values(by=["N_B", "N_A"], ascending=[False, False]).reset_index(drop=True)
    return out


def pairwise_pvalues_all_metrics_min(
    prepped_df: pd.DataFrame,
    *,
    group_col: str,
    id_col: str = "id_var",
    adjust: str = "holm",
    metrics=None,
):
    if metrics is None:
        metrics = [
            "Sensitivity",
            "Specificity",
            "PPV",
            "NPV",
            "Accuracy",
            "FPR",
            "FNR",
            "FDR",
            "FOR",
        ]

    all_out = []
    for m in metrics:
        dd, yy = _prep_metric_subset(prepped_df, m, group_col=group_col)
        if dd.empty or dd[yy].nunique() < 2:
            continue

        pairs = _ordered_pairs_by_frequency(dd, group_col)
        if not pairs:
            continue

        formula = f"{yy} ~ C({group_col})"
        try:
            res = fit_binomial_glm_with_cluster(dd, formula, id_col=id_col)
        except (PerfectSeparationError, PerfectSeparationWarning):
            warnings.warn(
                f"Skipping pairwise {m} inference for {group_col}: perfect separation detected.",
                RuntimeWarning,
            )
            continue

        out_m = _pairwise_from_fitted_model_min(
            res, dd=dd, group_col=group_col, y_col=yy, pairs=pairs, adjust=adjust
        )
        out_m.insert(0, "Metric", m)
        all_out.append(out_m)

    out = _concat_non_none(all_out)
    if out is None:
        return None

    keep = [
        "Metric",
        "Group_A",
        "Group_B",
        "Estimate",
        "Observed_Diff",
        "p",
        "N_A",
        "N_B",
        "reference",
        "p_adj",
        "Comparison",
    ]
    return (
        out[keep]
        .sort_values(by=["Metric", "N_B", "N_A"], ascending=[True, False, False])
        .reset_index(drop=True)
    )


# MAIN: build wide metric table + global pvals + pairwise table
#   Pairwise table includes Estimate:
#     - AUROC/AUPRC: mean bootstrap (A - B)
#     - Binomial metrics: risk difference (A - B)
# ============================================================
def build_groupwise_metrics_and_pvalues(
    df,
    group_col,
    id_col="id_var",
    B_auc=300,
    auc_perm_B=1000,
    pairwise_adjust="holm",
    min_n=MIN_GROUP_N,
):
    d = filter_group_analysis_df(df, group_col, min_n=min_n)
    d["y"] = to_binary01(d["final_outcome"])
    d["pred"] = to_binary01(d["prediction"])
    d = d.dropna(subset=["y", "pred"]).copy()
    d["y"] = d["y"].astype(int)
    d["pred"] = d["pred"].astype(int)
    if d.empty:
        return pd.DataFrame(columns=["Metric"]), pd.Series(dtype=float, name=group_col), None

    rows = []
    for level, gg in d.groupby(group_col, dropna=False, observed=False):
        if pd.isna(level):
            continue  # Skip NaN group; missing values should not be a group level
        r = compute_binomial_metrics_ci(gg, id_col=id_col)
        r["Level"] = str(level)
        rows.append(r)
    long = pd.DataFrame(rows)

    aucap_pairwise = None
    if "probability" in d.columns and d["probability"].notna().any():
        group_ci, aucap_pairwise = joint_cluster_bootstrap_auc_ap_minpairwise(
            d,
            group_col=group_col,
            id_col=id_col,
            score_col="probability",
            B=B_auc,
            seed=random_seed,
            pairwise_adjust=pairwise_adjust,
        )
        long["AUROC"] = long["Level"].map(lambda lvl: group_ci["AUROC"].get(str(lvl), None))
        long["AUPRC"] = long["Level"].map(lambda lvl: group_ci["AUPRC"].get(str(lvl), None))
    else:
        long["AUROC"] = None
        long["AUPRC"] = None

    wide = long.set_index("Level").T
    wide.insert(0, "Metric", wide.index)
    wide = wide[wide["Metric"] != "N"].reset_index(drop=True)

    pv = compute_group_difference_pvalues(
        d, group_col=group_col, id_col=id_col, auc_perm_B=auc_perm_B
    )
    pvals = pd.Series([pv.get(m, np.nan) for m in wide["Metric"].astype(str)], name=group_col)

    pairwise_tbl = pairwise_pvalues_all_metrics_min(
        d,
        group_col=group_col,
        id_col=id_col,
        adjust=pairwise_adjust,
    )
    pairwise_all = _concat_non_none([pairwise_tbl, aucap_pairwise])
    if pairwise_all is not None:
        keep = [
            "Metric",
            "Group_A",
            "Group_B",
            "Estimate",
            "Observed_Diff",
            "p",
            "N_A",
            "N_B",
            "reference",
            "p_adj",
            "Comparison",
        ]
        pairwise_all = pairwise_all[keep].reset_index(drop=True)

    return wide, pvals, pairwise_all


# ============================================================
# Example calls
# ============================================================
if has_race_ethnicity:
    race_table, race_pvals, race_pairwise = build_groupwise_metrics_and_pvalues(
        data,
        "race_ethnicity",
        id_col="id_var",
        B_auc=n_bootstrap,
        auc_perm_B=n_permutation,
        pairwise_adjust="holm",
    )
else:
    race_table, race_pvals, race_pairwise = None, None, None
if has_gender_identity:
    sex_table, sex_pvals, sex_pairwise = build_groupwise_metrics_and_pvalues(
        data,
        "gender_identity",
        id_col="id_var",
        B_auc=n_bootstrap,
        auc_perm_B=n_permutation,
        pairwise_adjust="holm",
    )
else:
    sex_table, sex_pvals, sex_pairwise = None, None, None
if has_age_group:
    age_table, age_pvals, age_pairwise = build_groupwise_metrics_and_pvalues(
        data,
        "age_group",
        id_col="id_var",
        B_auc=n_bootstrap,
        auc_perm_B=n_permutation,
        pairwise_adjust="holm",
    )
else:
    age_table, age_pvals, age_pairwise = None, None, None
if has_hpi_score:
    hpi_table, hpi_pvals, hpi_pairwise = build_groupwise_metrics_and_pvalues(
        data,
        "hpi_quartile",
        id_col="id_var",
        B_auc=n_bootstrap,
        auc_perm_B=n_permutation,
        pairwise_adjust="holm",
    )
else:
    hpi_table, hpi_pvals, hpi_pairwise = None, None, None

In [ ]:
def _format_p_value_display(p, *, digits=4, threshold=0.0001, na_str="NA"):
    if pd.isna(p):
        return na_str
    p = float(p)
    if p < threshold:
        return f"<{threshold:.{digits}f}"
    return f"{p:.{digits}f}"


def _format_estimate_ci_cell(cell, *, digits=3):
    if pd.isna(cell) or cell is None:
        return ""
    s = str(cell).strip()
    if s in {"", "None", "nan"}:
        return ""
    nums = re.findall(r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?", s)
    if "[" in s and len(nums) >= 3:
        return f"{float(nums[0]):.{digits}f} [{float(nums[1]):.{digits}f}, {float(nums[2]):.{digits}f}]"
    if "[NA, NA]" in s and len(nums) >= 1:
        return f"{float(nums[0]):.{digits}f} [NA, NA]"
    if len(nums) == 1:
        return f"{float(nums[0]):.{digits}f}"
    return s


def format_wide_table_for_display(wide_tbl: pd.DataFrame, *, digits=3) -> pd.DataFrame:
    if "Metric" not in wide_tbl.columns:
        raise ValueError("Expected column 'Metric' in wide table.")
    display_tbl = wide_tbl.copy()
    level_cols = [c for c in display_tbl.columns if c != "Metric"]
    keep_cols = [
        c
        for c in level_cols
        if display_tbl[c]
        .apply(lambda x: not (pd.isna(x) or str(x).strip() in {"", "None", "nan"}))
        .any()
    ]
    display_tbl = display_tbl[["Metric"] + keep_cols].copy()
    for col in keep_cols:
        display_tbl[col] = display_tbl[col].apply(
            lambda x: _format_estimate_ci_cell(x, digits=digits)
        )
    return display_tbl


def format_pvalue_table_for_display(tbl: pd.DataFrame) -> pd.DataFrame:
    display_tbl = tbl.copy()
    for col in [c for c in display_tbl.columns if c != "Metric"]:
        display_tbl[col] = pd.to_numeric(display_tbl[col], errors="coerce").map(
            lambda x: _format_p_value_display(x) if not pd.isna(x) else "NA"
        )
    return display_tbl


# ----------------------------
# 1) Wide -> stacked df_metrics
# ----------------------------
def wide_table_to_df_metrics(wide_tbl: pd.DataFrame, group_name: str) -> pd.DataFrame:
    if "Metric" not in wide_tbl.columns:
        raise ValueError("Expected column 'Metric' in wide table.")

    t = wide_tbl.copy()
    metric_names = t["Metric"].astype(str).tolist()
    level_cols = [c for c in t.columns if c != "Metric"]

    rows = [{"Group": group_name, **{m: np.nan for m in metric_names}}]
    metric_index = t["Metric"].astype(str)

    for lvl in level_cols:
        s = t.set_index(metric_index)[lvl]
        rows.append({"Group": str(lvl), **{m: s.get(m, np.nan) for m in metric_names}})

    return pd.DataFrame(rows)


dfs = []
if "race_table" in globals() and isinstance(race_table, pd.DataFrame):
    dfs.append(wide_table_to_df_metrics(race_table, "RACE/ETHNICITY"))
if "sex_table" in globals() and isinstance(sex_table, pd.DataFrame):
    dfs.append(wide_table_to_df_metrics(sex_table, "SEX"))
if "age_table" in globals() and isinstance(age_table, pd.DataFrame):
    dfs.append(wide_table_to_df_metrics(age_table, "AGE"))
if "hpi_table" in globals() and isinstance(hpi_table, pd.DataFrame):
    dfs.append(wide_table_to_df_metrics(hpi_table, "HPI"))
df_metrics = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(columns=["Group"])

# ----------------------------
# 2) Plotting helpers
# ----------------------------
group_colors = {"RACE/ETHNICITY": "blue", "SEX": "green", "AGE": "red", "HPI": "purple"}


def extract_metric_data(x):
    if pd.isna(x) or x == "None":
        return None, None, None
    try:
        value, ci = str(x).split(" [")
        lo, hi = ci.strip("]").split(", ")
        if lo == "NA" or hi == "NA":
            return float(value), None, None
        return float(value), float(lo), float(hi)
    except Exception:
        return None, None, None


def plot_metric(df, metric_name, ax):
    values, loerr, hierr, cats, cols = [], [], [], [], []
    group_label = None

    for _, row in df.iterrows():
        cell = row[metric_name]
        value, lcl, ucl = extract_metric_data(cell)

        if pd.isna(cell):
            group_label = row["Group"]
            continue
        if value is None:
            continue

        values.append(value)
        loerr.append(0.0 if lcl is None else value - lcl)
        hierr.append(0.0 if ucl is None else ucl - value)
        cats.append(row["Group"])
        cols.append(group_colors.get(group_label, "black"))

    for k, (val, le, ue, col) in enumerate(zip(values, loerr, hierr, cols)):
        ax.errorbar(k, val, yerr=[[le], [ue]], fmt="o", color=col, capsize=5)

    ax.grid(color="lightgray", linestyle="--", linewidth=0.5, alpha=0.7)
    ax.set_xticks(np.arange(len(cats)))
    ax.set_xticklabels(cats, rotation=45, ha="right")
    ax.set_ylabel(f"{metric_name} (with 95% CI)")
    ax.set_xlabel("")
    ax.set_title(f"{metric_name}")


# ----------------------------
# 3) Make 2-column figure + dedicated legend panel (last panel)
# ----------------------------
def plot_all_metric_estimates(df_metrics):
    metrics_to_plot = [c for c in df_metrics.columns if c not in ["Group", "N"]]
    n = len(metrics_to_plot)
    if n == 0:
        display(
            Markdown(
                "**No eligible demographic groups were available for the combined metric figure.**"
            )
        )
        return None

    K = n + 1
    ncols = 2
    nrows = int(np.ceil(K / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5 * nrows), constrained_layout=True)
    axes = np.array(axes).reshape(-1)
    for i, metric_name in enumerate(metrics_to_plot):
        plot_metric(df_metrics, metric_name, axes[i])

    legend_ax = axes[n]
    legend_ax.axis("off")
    handles = [
        plt.Line2D([0], [0], marker="o", color=color, label=group, linestyle="None")
        for group, color in group_colors.items()
    ]
    legend_ax.legend(
        handles=handles,
        labels=[h.get_label() for h in handles],
        loc="center",
        fontsize=12,
        frameon=False,
        ncol=1,
        handlelength=2.5,
        labelspacing=0.8,
        borderpad=1.0,
        title="Group Variable",
    )
    for j in range(n + 1, len(axes)):
        axes[j].axis("off")
    plt.show()
    return fig


_metric_figure = plot_all_metric_estimates(df_metrics)
if _metric_figure is not None:
    iteration_fig += 1
    display(
        Markdown(
            f"**Figure {iteration_fig}: Performance metrics [95% confidence intervals] by attribute groups.**"
        )
    )

#### Race/Ethnicity (Estimate CI)

In [ ]:
if has_race_ethnicity and isinstance(race_table, pd.DataFrame):
    display(HTML(format_wide_table_for_display(race_table).to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Race/ethnicity performance metrics estimates [95% confidence limits].**"
        )
    )
else:
    display(Markdown("**Race/ethnicity performance metrics estimates not available.**"))

#### Sex (Estimate CI)

In [ ]:
if has_gender_identity and isinstance(sex_table, pd.DataFrame):
    display(HTML(format_wide_table_for_display(sex_table).to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Sex performance metrics estimates [95% confidence limits].**"
        )
    )
else:
    display(Markdown("**Sex performance metrics estimates not available.**"))

#### Age (Estimate CI)

In [ ]:
if has_age_group and isinstance(age_table, pd.DataFrame):
    display(HTML(format_wide_table_for_display(age_table).to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Age performance metrics estimates [95% confidence limits].**"
        )
    )
else:
    display(Markdown("**Age performance metrics estimates not available.**"))

#### HPI (Estimate CI)

In [ ]:
if has_hpi_score and isinstance(hpi_table, pd.DataFrame):
    display(HTML(format_wide_table_for_display(hpi_table).to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: HPI performance metrics estimates [95% confidence limits].**"
        )
    )
else:
    display(Markdown("**HPI performance metrics estimates not available.**"))

## Task 2: Statistical Analyses of Performance Metric Differences

Within each attribute group, performance differences were statistically evaluated for AUROC, AUPRC, calibration intercept, calibration slope, sensitivity, specificity, positive predictive value, negative predictive value, false discovery rate, false omission rate, false positive rate, false negative rate, and accuracy. P-values for overall between-group differences within each attribute are shown in the following table. For every attribute with at least two eligible groups, pairwise comparisons were conducted to identify which groups differed. The Holm method was used to control the family-wise Type I error rate at 0.05 across pairwise comparisons within each metric. These results are provided in Appendix A.

### Statistical Analysis Results

In [ ]:
_metric_ref = next(
    (
        t
        for t in [race_table, sex_table, age_table, hpi_table]
        if isinstance(t, pd.DataFrame) and "Metric" in t.columns
    ),
    None,
)
if _metric_ref is not None:
    _pval_cols = {"Metric": _metric_ref["Metric"].astype(str)}
    if has_race_ethnicity and race_pvals is not None:
        _pval_cols["Race/Ethnicity"] = pd.Series(race_pvals)
    if has_gender_identity and sex_pvals is not None:
        _pval_cols["Sex"] = pd.Series(sex_pvals)
    if has_age_group and age_pvals is not None:
        _pval_cols["Age"] = pd.Series(age_pvals)
    if has_hpi_score and hpi_pvals is not None:
        _pval_cols["HPI"] = pd.Series(hpi_pvals)
    pval_table = pd.DataFrame(_pval_cols)
    display(HTML(format_pvalue_table_for_display(pval_table).to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Results (unadjusted p-values) of statistical analyses evaluating overall differential performance for attribute groups.**"
        )
    )
else:
    display(Markdown("**No demographic variables available for statistical analysis.**"))

## Task 3: Identification of Potentially Meaningful Performance Differences

In [ ]:
step3_text = f"""A commonly used metric for identifying potential disparities is the disparity ratio. The threshold for identifying differences of potential concern is the user-configurable `disparity_ratio` parameter in the configuration cell. In this analysis, the disparity ratio threshold is **{disparity_ratio:.0%}** (`disparity_ratio = {disparity_ratio:.2f}`), corresponding to a symmetric user-defined ratio band of **{ratio_low:.2f} to {ratio_high:.2f}** (`ratio_low = {ratio_low:.2f}`, `ratio_high = {ratio_high:.2f}`). This is a screening threshold, not the four-fifths rule or a universal fairness standard. For each performance metric, the disparity ratio is calculated for each group relative to every other group. Odds ratios are used for calibration intercept and calibration slope because these metrics are derived from logistic regression.

Statistically significant differences with disparity ratios outside the user-specified acceptable band are of potential concern for contributing to disparities, as are statistically significant odds ratios for calibration. These ratios are plotted for each performance metric with statistically significant differences highlighted. Statistical significance is determined using post-hoc pairwise comparisons adjusted for multiple comparisons (Appendix A)."""

display(Markdown(step3_text))

In [ ]:
# ============================================================
# Parsing helper: extract point estimate + CI bounds from strings
# ============================================================

_FLOAT_RE = r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?"


def _parse_ci(cell):
    """
    Parse a cell formatted like:
      "0.812 [0.750, 0.870]"
    Returns:
      (estimate, lcl, ucl)
    If only a single number exists, returns (estimate, nan, nan).
    """
    if pd.isna(cell):
        return np.nan, np.nan, np.nan
    if isinstance(cell, (int, float, np.floating)):
        return float(cell), np.nan, np.nan
    s = str(cell).replace("\n", " ").strip()
    if s == "" or s.lower() == "na":
        return np.nan, np.nan, np.nan
    nums = re.findall(_FLOAT_RE, s)
    if len(nums) >= 3:
        return float(nums[0]), float(nums[1]), float(nums[2])
    if len(nums) == 1:
        return float(nums[0]), np.nan, np.nan
    return np.nan, np.nan, np.nan


# ============================================================
# Figure: all pairwise disparity ratios by metric + legend panel
# ============================================================


def plot_allpair_ratio_panels_with_legend_panel(
    *,
    race_table: pd.DataFrame,
    data_for_counts: pd.DataFrame,
    race_col_in_data: str,
    metric_col: str = "Metric",
    pairwise_p_table: pd.DataFrame | None = None,  # expects Metric, Group_A, Group_B, p_adj (or p)
    p_col: str = "p_adj",
    alpha: float = 0.05,
    ncols: int = 3,
    figsize_per_panel=(6.5, 4.8),
    ratio_low: float = 0.8,
    ratio_high: float = 1.2,
    top_n: int | None = None,
    suptitle: str | None = None,
    compare_label_fmt: str = "{a} vs {b}",
    row_height: float = 0.35,  # increase to create more space between y labels
):
    """
    Create a panel plot (one panel per metric) showing ALL unique pairwise disparity ratios (A/B).

    - Ordering of groups is determined by their frequency in `data_for_counts[race_col_in_data]`.
    - Comparisons are generated as: for each anchor group B (most frequent → least),
      compare all less-frequent groups A against B.
    - Bar color:
        * orange if (adjusted) p-value < alpha for that metric/pair (from pairwise_p_table)
        * blue otherwise
    - Adds a final panel used solely for the legend.
    """

    # ----------------------------
    # Basic input checks
    # ----------------------------
    if metric_col not in race_table.columns:
        raise ValueError(f"race_table must include '{metric_col}' column.")
    if race_col_in_data not in data_for_counts.columns:
        raise ValueError(f"data_for_counts must include '{race_col_in_data}' column.")

    # ----------------------------
    # Determine subgroup column order (by frequency in underlying data)
    # ----------------------------
    level_cols = [c for c in race_table.columns if c != metric_col]

    counts = data_for_counts[race_col_in_data].astype(str).value_counts(dropna=False)
    ordered_levels = [lvl for lvl in counts.index.tolist() if lvl in level_cols] + [
        lvl for lvl in level_cols if lvl not in counts.index
    ]

    def _ordered_pairs(levels):
        """Return unique ordered pairs (A, B): A compared to anchor B."""
        out = []
        for b_i, b in enumerate(levels[:-1]):
            for a in levels[b_i + 1 :]:
                out.append((a, b))
        return out

    ordered_pairs = _ordered_pairs(ordered_levels)

    # ----------------------------
    # Parse metric table cells into numeric estimate/LCL/UCL
    # ----------------------------
    rt = race_table.copy()
    rt[metric_col] = rt[metric_col].astype(str)

    est = rt[[metric_col] + level_cols].copy()
    lcl = rt[[metric_col] + level_cols].copy()
    ucl = rt[[metric_col] + level_cols].copy()
    for c in level_cols:
        parsed = rt[c].apply(_parse_ci)
        est[c] = parsed.map(lambda x: x[0])
        lcl[c] = parsed.map(lambda x: x[1])
        ucl[c] = parsed.map(lambda x: x[2])

    metrics = est[metric_col].tolist()

    # ----------------------------
    # Build p-value lookup for coloring bars by significance
    #   key: (Metric, sorted(A,B)) -> p_adj (or p)
    # ----------------------------
    p_lookup = {}
    if pairwise_p_table is not None and len(pairwise_p_table):
        req = {"Metric", "Group_A", "Group_B"}
        if not req.issubset(set(pairwise_p_table.columns)):
            raise ValueError(
                "pairwise_p_table must include columns: Metric, Group_A, Group_B (and p_col)."
            )
        if p_col not in pairwise_p_table.columns:
            raise ValueError(f"pairwise_p_table must include '{p_col}' (or set p_col=...).")

        ppt = pairwise_p_table.copy()
        ppt["Metric"] = ppt["Metric"].astype(str)
        ppt["Group_A"] = ppt["Group_A"].astype(str)
        ppt["Group_B"] = ppt["Group_B"].astype(str)
        ppt[p_col] = pd.to_numeric(ppt[p_col], errors="coerce")

        for _, r in ppt.iterrows():
            m = r["Metric"]
            a = r["Group_A"]
            b = r["Group_B"]
            key = (m, tuple(sorted([a, b])))
            pv = r[p_col]
            if key not in p_lookup or (np.isfinite(pv) and pv < p_lookup[key]):
                p_lookup[key] = float(pv) if np.isfinite(pv) else np.nan

    # ----------------------------
    # Auto-scale figure height based on how many bars exist per panel
    # ----------------------------
    max_bars = 0
    for metric in metrics:
        e_row = est.loc[est[metric_col] == metric].iloc[0]
        n = 0
        for a, b in ordered_pairs:
            ea = float(e_row.get(a, np.nan))
            eb = float(e_row.get(b, np.nan))
            if (np.isfinite(ea) and np.isfinite(eb)) and eb != 0:
                n += 1
        if top_n is not None:
            n = min(n, int(top_n))
        max_bars = max(max_bars, n)

    # reserve 1 panel for legend
    K = len(metrics) + 1
    nrows = int(np.ceil(K / ncols))
    fig_w = figsize_per_panel[0] * ncols
    fig_h = max(figsize_per_panel[1] * nrows, row_height * max_bars * nrows)

    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols, figsize=(fig_w, fig_h), constrained_layout=True
    )
    axes = np.array(axes).reshape(-1)

    # color mapping
    c_within = "tab:blue"
    c_outside = "red"
    legend_handles = [
        Patch(facecolor=c_outside, label=f"Outside [{ratio_low}, {ratio_high}] & significant"),
        Patch(facecolor=c_within, label=f"Within [{ratio_low}, {ratio_high}] or not significant"),
        Line2D([0], [0], color="black", linestyle="--", linewidth=2, label="Ratio = 1"),
        Line2D(
            [0],
            [0],
            color="gray",
            linestyle=":",
            linewidth=2,
            alpha=0.7,
            label=f"Disparity ratio limits [{ratio_low}, {ratio_high}]",
        ),
    ]
    # ----------------------------
    # Draw one bar chart panel per metric
    # ----------------------------
    for i, metric in enumerate(metrics):
        ax = axes[i]

        e_row = est.loc[est[metric_col] == metric].iloc[0]
        l_row = lcl.loc[lcl[metric_col] == metric].iloc[0]
        u_row = ucl.loc[ucl[metric_col] == metric].iloc[0]

        comps, ratio_pt = [], []
        ratio_lcl, ratio_ucl = [], []
        pvals = []

        # Build ordered pairwise ratios A/B
        for a, b in ordered_pairs:
            ea = float(e_row.get(a, np.nan))
            eb = float(e_row.get(b, np.nan))
            if not (np.isfinite(ea) and np.isfinite(eb)) or eb == 0:
                continue

            la = float(l_row.get(a, np.nan))
            ua = float(u_row.get(a, np.nan))
            lb = float(l_row.get(b, np.nan))
            ub = float(u_row.get(b, np.nan))

            r = ea / eb
            comps.append(compare_label_fmt.format(a=a, b=b))
            ratio_pt.append(r)

            # Approx ratio CI from group CIs (conservative bounds)
            if all(np.isfinite([la, ua, lb, ub])) and (lb > 0) and (ub > 0):
                r_l = la / ub
                r_u = ua / lb
            else:
                r_l, r_u = np.nan, np.nan
            ratio_lcl.append(r_l)
            ratio_ucl.append(r_u)

            key = (str(metric), tuple(sorted([str(a), str(b)])))
            pvals.append(p_lookup.get(key, np.nan))

        dfp = pd.DataFrame(
            {
                "Comparison": comps,
                "Ratio": ratio_pt,
                "Ratio_LCL": ratio_lcl,
                "Ratio_UCL": ratio_ucl,
                p_col: pvals,
            }
        )

        if top_n is not None:
            dfp = dfp.head(int(top_n)).copy()

        # Determine color
        ratio_vals = pd.to_numeric(dfp["Ratio"], errors="coerce")
        pvals_arr = pd.to_numeric(dfp[p_col], errors="coerce")
        outside_interval = ~ratio_vals.between(ratio_low, ratio_high, inclusive="both")
        significant = pvals_arr < alpha
        is_red = (outside_interval & significant).fillna(False).to_numpy(dtype=bool)
        c_default = "tab:blue"
        c_red = "red"
        colors = np.where(is_red, c_red, c_default)

        y = np.arange(len(dfp))
        ax.barh(y, dfp["Ratio"].values, color=colors)

        # Reference lines: ratio=1 (black) and limits (gray dotted)
        ax.axvline(1.0, linestyle="--", linewidth=2.5, color="black", alpha=0.9)
        ax.axvline(ratio_low, linestyle=":", linewidth=2, color="gray", alpha=0.7)
        ax.axvline(ratio_high, linestyle=":", linewidth=2, color="gray", alpha=0.7)

        ax.set_title(metric)
        ax.set_xlabel("Disparity ratio (A / B)")
        ax.grid(axis="x", linestyle="--", alpha=0.3)
        ax.ticklabel_format(axis="x", style="plain", useOffset=False)
        ax.invert_yaxis()

        if (i % ncols) == 0:
            ax.set_yticks(y)
            ax.set_yticklabels(dfp["Comparison"])
        else:
            ax.set_yticks(y)
            ax.set_yticklabels([])
            ax.tick_params(axis="y", length=0)

    # ----------------------------
    # Legend-only panel
    # ----------------------------
    legend_ax = axes[len(metrics)]
    legend_ax.axis("off")
    legend_ax.legend(
        handles=legend_handles,
        loc="center",
        frameon=False,
        ncol=1,
        fontsize=12,
        handlelength=2.5,
        labelspacing=0.8,
        borderpad=1.0,
    )
    legend_ax.set_title("LEGEND", pad=10)

    # Hide any extra axes (if grid larger than needed)
    for j in range(len(metrics) + 1, len(axes)):
        axes[j].axis("off")

    if suptitle is not None:
        fig.suptitle(suptitle, fontsize=14)

    plt.show()


# ============================================================
# Calibration disparity model: pairwise ORs for intercept and slope
# ============================================================


def calibration_pairwise_or(
    df,
    y_col,
    prob_col,
    race_col,
    p_adjust="holm",
    min_n_per_group=MIN_GROUP_N,
    z0=None,
    id_col="id_var",
    prob_clip=0.01,
):
    """Fit group-specific calibration curves and return pairwise OR tables."""
    from patsy import build_design_matrices, dmatrix

    table_columns = [
        "Comparison",
        "Group_A",
        "Group_B",
        "Race_A",
        "Race_B",
        "logOR",
        "SE",
        "z",
        "p",
        "OR",
        "LCL",
        "UCL",
        "Observed Difference",
        "p_adj",
        "N_A",
        "N_B",
    ]

    def empty_table():
        return pd.DataFrame(columns=table_columns)

    required = [y_col, prob_col, race_col]
    use_cluster = id_col is not None
    if use_cluster and id_col not in df.columns:
        raise KeyError(f"Missing required calibration cluster column: {id_col}")
    if use_cluster:
        required.append(id_col)
    required = list(dict.fromkeys(required))
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required calibration columns: {missing}")

    d = df[required].copy().dropna(subset=required)
    d["_cal_y"] = pd.to_numeric(d[y_col], errors="coerce")
    d["_cal_prob"] = pd.to_numeric(d[prob_col], errors="coerce")
    d = d.loc[
        d["_cal_y"].isin([0, 1])
        & np.isfinite(d["_cal_prob"])
        & d["_cal_prob"].between(0, 1, inclusive="both")
    ].copy()
    d["_cal_group"] = d[race_col].astype(str).str.strip()
    d = d.loc[d["_cal_group"].ne("")].copy()

    eligibility = d.groupby("_cal_group", observed=True)["_cal_y"].agg(
        N="size", OutcomeClasses="nunique"
    )
    eligibility = eligibility.sort_values("N", ascending=False)
    keep_levels = eligibility.index[
        (eligibility["N"] >= int(min_n_per_group)) & (eligibility["OutcomeClasses"] == 2)
    ].tolist()
    d = d.loc[d["_cal_group"].isin(keep_levels)].copy()
    counts = d["_cal_group"].value_counts()
    if len(keep_levels) < 2:
        baseline = keep_levels[0] if keep_levels else None
        return None, baseline, empty_table(), empty_table()

    baseline = keep_levels[0]
    levels = [baseline] + [x for x in keep_levels if x != baseline]
    d["_cal_group"] = pd.Categorical(d["_cal_group"], categories=levels, ordered=True)

    eps = float(prob_clip)
    if not 0 < eps < 0.5:
        raise ValueError("prob_clip must be between 0 and 0.5.")
    clipped = d["_cal_prob"].clip(eps, 1.0 - eps)
    d["_cal_z"] = np.log(clipped / (1.0 - clipped))

    glm = smf.glm(
        formula="_cal_y ~ _cal_z * C(_cal_group)",
        data=d,
        family=sm.families.Binomial(),
    )
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error", PerfectSeparationWarning)
            if use_cluster and d[id_col].nunique(dropna=True) > 1:
                model = glm.fit(
                    cov_type="cluster",
                    cov_kwds={"groups": d[id_col].astype(str).to_numpy()},
                )
            else:
                model = glm.fit()
    except (PerfectSeparationError, PerfectSeparationWarning):
        warnings.warn(
            f"Calibration pairwise inference skipped for {race_col}: perfect separation detected.",
            RuntimeWarning,
        )
        return None, baseline, empty_table(), empty_table()

    params = np.asarray(model.params, dtype=float)
    covariance = np.asarray(model.cov_params(), dtype=float)
    design_info = dmatrix("_cal_z * C(_cal_group)", data=d).design_info

    def design_row(group, z_value):
        new_data = pd.DataFrame(
            {
                "_cal_z": [float(z_value)],
                "_cal_group": pd.Categorical([group], categories=levels, ordered=True),
            }
        )
        matrix = build_design_matrices([design_info], new_data)[0]
        return np.asarray(matrix, dtype=float).reshape(-1)

    def contrast_result(vector):
        vector = np.asarray(vector, dtype=float)
        estimate = float(vector @ params)
        variance = float(vector @ covariance @ vector)
        if variance < 0 and np.isclose(variance, 0.0):
            variance = 0.0
        se = float(np.sqrt(variance)) if variance >= 0 else np.nan
        if np.isfinite(se) and se > 0:
            statistic = estimate / se
            p_value = float(2.0 * stats.norm.sf(abs(statistic)))
        elif np.isfinite(se) and np.isclose(estimate, 0.0):
            statistic, p_value = 0.0, 1.0
        else:
            statistic, p_value = np.nan, np.nan
        return estimate, se, statistic, p_value

    def safe_exp(value):
        return float(np.exp(np.clip(value, -700.0, 700.0)))

    def inv_logit(value):
        value = float(np.clip(value, -700.0, 700.0))
        return float(1.0 / (1.0 + np.exp(-value)))

    if z0 is None:
        threshold = float(globals().get("classification_threshold", 0.5))
        threshold = float(np.clip(threshold, eps, 1.0 - eps))
        z0 = float(np.log(threshold / (1.0 - threshold)))
    else:
        z0 = float(z0)

    components = {}
    for group in levels:
        x0 = design_row(group, 0.0)
        slope_vector = design_row(group, 1.0) - x0
        components[group] = {
            "intercept_vector": x0,
            "slope_vector": slope_vector,
            "intercept": float(x0 @ params),
            "slope": float(slope_vector @ params),
        }

    pairs = [(a, b) for i, b in enumerate(levels[:-1]) for a in levels[i + 1 :]]
    rows_int, rows_slope = [], []

    for a, b in pairs:
        comp_a, comp_b = components[a], components[b]
        p_a = inv_logit(comp_a["intercept"] + comp_a["slope"] * z0)
        p_b = inv_logit(comp_b["intercept"] + comp_b["slope"] * z0)

        int_vector = comp_a["intercept_vector"] - comp_b["intercept_vector"]
        est, se, statistic, p_value = contrast_result(int_vector)
        rows_int.append(
            {
                "Comparison": f"{a} -- {b}",
                "Group_A": a,
                "Group_B": b,
                "Race_A": a,
                "Race_B": b,
                "logOR": est,
                "SE": se,
                "z": statistic,
                "p": p_value,
                "OR": safe_exp(est),
                "LCL": safe_exp(est - 1.96 * se) if np.isfinite(se) else np.nan,
                "UCL": safe_exp(est + 1.96 * se) if np.isfinite(se) else np.nan,
                "Observed Difference": float(p_a - p_b),
            }
        )

        slope_vector = comp_a["slope_vector"] - comp_b["slope_vector"]
        est, se, statistic, p_value = contrast_result(slope_vector)
        derivative_a = p_a * (1.0 - p_a) * comp_a["slope"]
        derivative_b = p_b * (1.0 - p_b) * comp_b["slope"]
        rows_slope.append(
            {
                "Comparison": f"{a} -- {b}",
                "Group_A": a,
                "Group_B": b,
                "Race_A": a,
                "Race_B": b,
                "logOR": est,
                "SE": se,
                "z": statistic,
                "p": p_value,
                "OR": safe_exp(est),
                "LCL": safe_exp(est - 1.96 * se) if np.isfinite(se) else np.nan,
                "UCL": safe_exp(est + 1.96 * se) if np.isfinite(se) else np.nan,
                "Observed Difference": float(derivative_a - derivative_b),
            }
        )

    tbl_int = pd.DataFrame(rows_int)
    tbl_slope = pd.DataFrame(rows_slope)
    p_all = np.concatenate(
        [
            pd.to_numeric(tbl_int["p"], errors="coerce").to_numpy(),
            pd.to_numeric(tbl_slope["p"], errors="coerce").to_numpy(),
        ]
    )
    p_adj_all = np.full(len(p_all), np.nan, dtype=float)
    finite_p = np.isfinite(p_all)
    if finite_p.any():
        p_adj_all[finite_p] = multipletests(p_all[finite_p], method=p_adjust)[1]
    tbl_int["p_adj"] = p_adj_all[: len(tbl_int)]
    tbl_slope["p_adj"] = p_adj_all[len(tbl_int) :]

    for table in (tbl_int, tbl_slope):
        table["N_A"] = table["Group_A"].map(counts)
        table["N_B"] = table["Group_B"].map(counts)
        table.sort_values(
            ["N_B", "N_A"],
            ascending=[False, False],
            kind="mergesort",
            inplace=True,
        )
        table.reset_index(drop=True, inplace=True)

    return model, baseline, tbl_int[table_columns], tbl_slope[table_columns]


# ============================================================
# Visualization: paired forest plots (intercept + slope)
# ============================================================


def combined_forest_plot(tbl_int, tbl_slope, sig_level=0.05):
    """Create paired forest plots for calibration intercept and slope ORs."""
    required = {"Comparison", "OR", "LCL", "UCL", "p_adj"}
    valid_tables = (
        isinstance(tbl_int, pd.DataFrame)
        and isinstance(tbl_slope, pd.DataFrame)
        and required.issubset(tbl_int.columns)
        and required.issubset(tbl_slope.columns)
        and len(tbl_int) > 0
        and len(tbl_slope) > 0
    )
    if not valid_tables:
        fig, ax = plt.subplots(figsize=(10, 2.5))
        ax.axis("off")
        ax.text(
            0.5,
            0.5,
            "Calibration pairwise comparisons require at least two eligible groups.",
            ha="center",
            va="center",
            wrap=True,
        )
        return fig, (ax,)

    intercept = tbl_int.copy().reset_index(drop=True)
    slope = tbl_slope.set_index("Comparison").reindex(intercept["Comparison"]).reset_index()
    y_labels = intercept["Comparison"].astype(str).tolist()
    y = np.arange(len(y_labels))

    fig, (ax1, ax2) = plt.subplots(
        ncols=2,
        sharey=True,
        figsize=(12, max(6, 0.35 * len(y_labels))),
    )

    bound_values = [1.0]
    for table in (intercept, slope):
        for col in ("OR", "LCL", "UCL"):
            values = pd.to_numeric(table[col], errors="coerce").to_numpy()
            bound_values.extend(values[np.isfinite(values) & (values > 0)].tolist())
    xmin = max(0.2, min(bound_values) * 0.95)
    xmax = min(5.0, max(bound_values) * 1.05)
    if xmin >= xmax:
        xmin, xmax = 0.2, 5.0

    def draw(ax, table, title):
        odds = pd.to_numeric(table["OR"], errors="coerce").to_numpy()
        lower = pd.to_numeric(table["LCL"], errors="coerce").to_numpy()
        upper = pd.to_numeric(table["UCL"], errors="coerce").to_numpy()
        adjusted_p = pd.to_numeric(table["p_adj"], errors="coerce").to_numpy()
        significant = np.isfinite(adjusted_p) & (adjusted_p < sig_level)

        for mask, color, label in (
            (~significant, "C0", "Not significant"),
            (significant, "C1", f"Significant (p < {sig_level})"),
        ):
            ci_mask = mask & np.isfinite(lower) & np.isfinite(upper)
            point_mask = mask & np.isfinite(odds)
            ax.hlines(y[ci_mask], lower[ci_mask], upper[ci_mask], linewidth=2, color=color)
            ax.scatter(odds[point_mask], y[point_mask], s=35, color=color, label=label)

        ax.axvline(1.0, linestyle="--", color="black", linewidth=1.5)
        ax.set_xlim(xmin, xmax)
        ax.set_title(title)
        ax.set_xlabel("Odds ratio")
        ax.grid(axis="x", linestyle="--", alpha=0.3)
        ax.ticklabel_format(style="plain", axis="x")
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, pos: f"{x:g}"))

    draw(ax1, intercept, "Calibration Intercept")
    draw(ax2, slope, "Calibration Slope")
    ax1.set_yticks(y)
    ax1.set_yticklabels(y_labels)
    ax1.invert_yaxis()

    handles, labels = ax2.get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=2,
        frameon=False,
        bbox_to_anchor=(0.5, -0.01),
    )
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    return fig, (ax1, ax2)

### Performance Disparities by Attribute Group

#### Race/Ethnicity (Disparity)

In [ ]:
if has_race_ethnicity:
    # ============================================================
    # run
    # ============================================================

    model, baseline, tbl_int_race, tbl_slope_race = calibration_pairwise_or(
        df=data,
        y_col="final_outcome",
        prob_col="probability",
        race_col="race_ethnicity",
        p_adjust="holm",
        id_col="id_var",
    )

    combined_forest_plot(tbl_int_race, tbl_slope_race)
    plt.show()

    iteration_fig += 1
    display(
        Markdown(
            f"**Figure {iteration_fig}: Pairwise racial/ethnic differences in model calibration (intercept and slope; Holm-adjusted p-values).**"
        )
    )
else:
    tbl_int_race, tbl_slope_race = None, None
    display(
        Markdown("**Race/ethnicity data not available, skipping calibration disparity panel.**")
    )

In [ ]:
if has_race_ethnicity and isinstance(race_table, pd.DataFrame):
    # ============================================================
    # RUN: pairwise disparity ratio panels
    # ============================================================
    plot_allpair_ratio_panels_with_legend_panel(
        race_table=race_table,
        data_for_counts=data,
        race_col_in_data="race_ethnicity",
        metric_col="Metric",
        pairwise_p_table=race_pairwise,
        p_col="p_adj",
        alpha=0.05,
        ncols=3,
        ratio_low=ratio_low,
        ratio_high=ratio_high,
        top_n=None,
        suptitle="",
        compare_label_fmt="{a} vs {b}",
        row_height=0.25,
    )
    plt.show()

    iteration_fig += 1
    display(
        Markdown(
            f"**Figure {iteration_fig}: Race/ethnicity pairwise disparity ratios across performance metrics (Holm-adjusted p-values).**"
        )
    )

#### Sex (Disparity)

In [ ]:
if has_gender_identity:
    model, baseline, tbl_int_sex, tbl_slope_sex = calibration_pairwise_or(
        df=data,
        y_col="final_outcome",
        prob_col="probability",
        race_col="gender_identity",
        p_adjust="holm",
        id_col="id_var",
    )

    combined_forest_plot(tbl_int_sex, tbl_slope_sex)
    plt.show()

    iteration_fig += 1
    display(
        Markdown(
            f"**Figure {iteration_fig}: Pairwise sex differences in model calibration (intercept and slope; Holm-adjusted p-values).**"
        )
    )
else:
    tbl_int_sex, tbl_slope_sex = None, None
    display(Markdown("**Sex/gender data not available, skipping calibration disparity panel.**"))

In [ ]:
if has_gender_identity and isinstance(sex_table, pd.DataFrame):
    fig = plot_allpair_ratio_panels_with_legend_panel(
        race_table=sex_table,
        data_for_counts=data,
        race_col_in_data="gender_identity",
        metric_col="Metric",
        pairwise_p_table=sex_pairwise,
        p_col="p_adj",
        alpha=0.05,
        ncols=3,
        ratio_low=ratio_low,
        ratio_high=ratio_high,
        top_n=None,
        suptitle="",
        compare_label_fmt="{a} vs {b}",
        row_height=0.25,
    )

    iteration_fig += 1
    display(
        Markdown(
            f"**Figure {iteration_fig}: Sex pairwise disparity ratios across performance metrics (Holm-adjusted p-values).**"
        )
    )

#### Age (Disparity)

In [ ]:
if has_age_group:
    model, baseline, tbl_int_age, tbl_slope_age = calibration_pairwise_or(
        df=data,
        y_col="final_outcome",
        prob_col="probability",
        race_col="age_group",
        p_adjust="holm",
        id_col="id_var",
    )

    combined_forest_plot(tbl_int_age, tbl_slope_age)
    plt.show()

    iteration_fig += 1
    display(
        Markdown(
            f"**Figure {iteration_fig}: Pairwise age group differences in model calibration (intercept and slope; Holm-adjusted p-values).**"
        )
    )
else:
    tbl_int_age, tbl_slope_age = None, None
    display(Markdown("**Age group data not available, skipping calibration disparity panel.**"))

In [ ]:
if has_age_group and isinstance(age_table, pd.DataFrame):
    fig = plot_allpair_ratio_panels_with_legend_panel(
        race_table=age_table,
        data_for_counts=data,
        race_col_in_data="age_group",
        metric_col="Metric",
        pairwise_p_table=age_pairwise,
        p_col="p_adj",
        alpha=0.05,
        ncols=3,
        ratio_low=ratio_low,
        ratio_high=ratio_high,
        top_n=None,
        suptitle="",
        compare_label_fmt="{a} vs {b}",
        row_height=0.25,
    )

    iteration_fig += 1
    display(
        Markdown(
            f"**Figure {iteration_fig}: Age group pairwise disparity ratios across performance metrics (Holm-adjusted p-values).**"
        )
    )

#### HPI (Disparity)

In [ ]:
if has_hpi_score:
    model, baseline, tbl_int_hpi, tbl_slope_hpi = calibration_pairwise_or(
        df=data,
        y_col="final_outcome",
        prob_col="probability",
        race_col="hpi_quartile",
        p_adjust="holm",
        id_col="id_var",
    )

    combined_forest_plot(tbl_int_hpi, tbl_slope_hpi)
    plt.show()

    iteration_fig += 1
    display(
        Markdown(
            f"**Figure {iteration_fig}: Pairwise hpi quantile differences in model calibration (intercept and slope; Holm-adjusted p-values).**"
        )
    )
else:
    tbl_int_hpi, tbl_slope_hpi = None, None
    display(Markdown("**HPI score data not available, skipping calibration disparity panel.**"))

In [ ]:
if has_hpi_score and isinstance(hpi_table, pd.DataFrame):
    fig = plot_allpair_ratio_panels_with_legend_panel(
        race_table=hpi_table,
        data_for_counts=data,
        race_col_in_data="hpi_quartile",
        metric_col="Metric",
        pairwise_p_table=hpi_pairwise,
        p_col="p_adj",
        alpha=0.05,
        ncols=3,
        ratio_low=ratio_low,
        ratio_high=ratio_high,
        top_n=None,
        suptitle="",
        compare_label_fmt="{a} vs {b}",
        row_height=0.25,
    )

    iteration_fig += 1
    display(
        Markdown(
            f"**Figure {iteration_fig}: HPI quantile pairwise disparity ratios across performance metrics (Holm-adjusted p-values).**"
        )
    )

### Disparity Ratio Summarization

The following metrics had statistically significant pairwise differences (after p-value adjustment) and exceeded the user-specified disparity threshold (or had an odds ratio significantly different from 1 for calibration slope or intercept) for at least one group comparison.

In [ ]:
display(
    Markdown(
        f"The disparity ratio threshold used was **{disparity_ratio:.0%}** "
        f"(acceptable ratio band: **{ratio_low:.2f} to {ratio_high:.2f}**). "
        f"Statistical significance is based on Holm-adjusted pairwise p-values."
    )
)

In [ ]:
# ============================================================
# Summarize disparities across variables and metrics
#   SIGNIFICANCE RULE (your requirement):
#     A metric is "significant" for a variable if ANY pairwise p_adj
#     for that metric (in pairwise_p_tables_by_var[var]) is < sig_level.
# ============================================================


def summarize_disparities_all_vars_pairwise_any(
    *,
    data: pd.DataFrame,
    tables_by_var: dict,  # {var_name: wide metric table}
    group_cols_by_var: dict,  # {var_name: column in `data` for ordering levels}
    metric_col: str = "Metric",
    ratio_low: float = 0.8,
    ratio_high: float = 1.2,
    sig_level: float = 0.05,
    top_k_examples_per_metric: int = 5,
    calibration_tables_by_var: dict | None = None,  # {var_name: {"Intercept": df, "Slope": df}}
    pairwise_p_tables_by_var: (
        dict | None
    ) = None,  # {var_name: df with Metric, Group_A, Group_B, p_adj (or p)}
    pairwise_p_col: str = "p_adj",
):
    """
    Performance metric is kept for var_name if BOTH:
      (1) ANY pairwise p_adj for that metric is < sig_level
      (2) exists at least one pair (A,B) such that:
            - same pair has p_adj < sig_level
            - ratio estA/estB is outside [ratio_low, ratio_high]
          CI non-overlap is not required because the adjusted pairwise test supplies
          the statistical-significance criterion.

    Calibration (optional):
      For each var part (Intercept/Slope), flag if ANY row has:
        p_adj < sig_level AND (LCL>1 OR UCL<1)

    Returns:
      summary_df, metric_list_by_var
    """

    # ----------------------------
    # Pairwise lookup per variable:
    #   lu[(metric, sorted(A,B))] = min p_adj
    # and minp_metric[metric] = min p_adj across all pairs for that metric
    # ----------------------------
    pairwise_lookup_by_var = {}
    metric_min_p_by_var = {}

    if not isinstance(pairwise_p_tables_by_var, dict) or len(pairwise_p_tables_by_var) == 0:
        raise ValueError("pairwise_p_tables_by_var is required for the 'ANY pairwise p_adj' rule.")

    for var_name, tbl in pairwise_p_tables_by_var.items():
        if not isinstance(tbl, pd.DataFrame) or len(tbl) == 0:
            continue

        needed = {metric_col, "Group_A", "Group_B", pairwise_p_col}
        if not needed.issubset(set(tbl.columns)):
            raise ValueError(
                f"Pairwise p table for '{var_name}' must include columns: {sorted(list(needed))}"
            )

        t = tbl[[metric_col, "Group_A", "Group_B", pairwise_p_col]].copy()
        t[metric_col] = t[metric_col].astype(str)
        t["Group_A"] = t["Group_A"].astype(str)
        t["Group_B"] = t["Group_B"].astype(str)
        t[pairwise_p_col] = pd.to_numeric(t[pairwise_p_col], errors="coerce")

        lu = {}
        minp = {}

        for _, r in t.iterrows():
            m = r[metric_col]
            a = r["Group_A"]
            b = r["Group_B"]
            p = r[pairwise_p_col]

            key = (m, tuple(sorted([a, b])))

            if key not in lu or (np.isfinite(p) and p < lu[key]):
                lu[key] = float(p) if np.isfinite(p) else np.nan

            if np.isfinite(p):
                if m not in minp or p < minp[m]:
                    minp[m] = float(p)

        pairwise_lookup_by_var[var_name] = lu
        metric_min_p_by_var[var_name] = minp

    # ----------------------------
    # ordering helper
    # ----------------------------
    def ordered_pairs(levels):
        out = []
        for b_i, b in enumerate(levels[:-1]):
            for a in levels[b_i + 1 :]:
                out.append((a, b))
        return out

    rows = []
    metric_list_by_var = {}

    # ----------------------------
    # Main loop over variables
    # ----------------------------
    for var_name, wide_tbl in tables_by_var.items():
        if not isinstance(wide_tbl, pd.DataFrame) or len(wide_tbl) == 0:
            continue
        if metric_col not in wide_tbl.columns:
            raise ValueError(f"Wide table for '{var_name}' must include '{metric_col}' column.")
        if var_name not in group_cols_by_var:
            raise ValueError(f"group_cols_by_var missing entry for '{var_name}'.")
        if var_name not in pairwise_lookup_by_var:
            raise ValueError(f"pairwise_p_tables_by_var missing table for '{var_name}'.")

        group_col = group_cols_by_var[var_name]
        level_cols = [c for c in wide_tbl.columns if c != metric_col]

        counts = data[group_col].astype(str).value_counts(dropna=False)
        ordered_levels = [lvl for lvl in counts.index.tolist() if lvl in level_cols] + [
            lvl for lvl in level_cols if lvl not in counts.index
        ]
        pairs_order = ordered_pairs(ordered_levels)

        rt = wide_tbl.copy()
        rt[metric_col] = rt[metric_col].astype(str)

        est = rt[[metric_col] + level_cols].copy()
        lcl = rt[[metric_col] + level_cols].copy()
        ucl = rt[[metric_col] + level_cols].copy()

        for c in level_cols:
            parsed = rt[c].apply(_parse_ci)
            est[c] = parsed.map(lambda x: x[0])
            lcl[c] = parsed.map(lambda x: x[1])
            ucl[c] = parsed.map(lambda x: x[2])

        metrics = est[metric_col].tolist()

        pairwise_lu = pairwise_lookup_by_var[var_name]
        metric_minp = metric_min_p_by_var.get(var_name, {})

        keep_metrics = []

        for metric in metrics:
            min_pair_p = metric_minp.get(str(metric), np.nan)
            is_sig_metric = bool(np.isfinite(min_pair_p) and min_pair_p < sig_level)

            e_row = est.loc[est[metric_col] == metric].iloc[0]
            l_row = lcl.loc[lcl[metric_col] == metric].iloc[0]
            u_row = ucl.loc[ucl[metric_col] == metric].iloc[0]

            violations = []

            for a, b in pairs_order:
                ea, eb = float(e_row.get(a, np.nan)), float(e_row.get(b, np.nan))
                la, ua = float(l_row.get(a, np.nan)), float(u_row.get(a, np.nan))
                lb, ub = float(l_row.get(b, np.nan)), float(u_row.get(b, np.nan))

                if not (np.isfinite(ea) and np.isfinite(eb)) or eb == 0:
                    continue

                r = ea / eb
                within_band = (r >= ratio_low) and (r <= ratio_high)

                key = (str(metric), tuple(sorted([str(a), str(b)])))
                pair_p = pairwise_lu.get(key, np.nan)
                pair_sig = bool(np.isfinite(pair_p) and pair_p < sig_level)

                # Flag if statistically significant AND outside the disparity band.
                # CI non-overlap is NOT required: the pairwise p-value already encodes
                # statistical significance, and requiring non-overlap is a redundant,
                # overly conservative criterion that suppresses valid disparity findings.
                if pair_sig and (not within_band):
                    violations.append((a, b, r, la, ua, lb, ub, pair_p))

            any_violation = len(violations) > 0
            meets_both = is_sig_metric and any_violation

            if meets_both:
                keep_metrics.append(metric)

            examples = []
            if any_violation:
                dev = np.array([max(v[2], 1.0 / v[2]) for v in violations], dtype=float)
                order = np.argsort(-dev)
                for k in order[:top_k_examples_per_metric]:
                    a, b, r, la, ua, lb, ub, pair_p = violations[int(k)]
                    examples.append(
                        f"{a} vs {b}: ratio={r:.3f}; "
                        f"{a} CI=[{la:.3f},{ua:.3f}] vs {b} CI=[{lb:.3f},{ub:.3f}]; "
                        f"{pairwise_p_col}={pair_p:.3g}"
                    )

            rows.append(
                {
                    "Variable": var_name,
                    "Metric": metric,
                    "Type": "Performance",
                    "min_pairwise_p_adj_for_metric": min_pair_p,
                    "Significant": is_sig_metric,
                    "Any_pair_violation": any_violation,
                    "Meets_both": meets_both,
                    "Top_violation_examples": "; ".join(examples),
                }
            )

        metric_list_by_var[var_name] = keep_metrics

    # ----------------------------
    # Calibration metrics (optional)
    # ----------------------------
    if isinstance(calibration_tables_by_var, dict) and len(calibration_tables_by_var) > 0:
        for var_name, parts in calibration_tables_by_var.items():
            if not isinstance(parts, dict):
                continue

            keep_cal = []

            for part_name, cal_df in parts.items():
                if not isinstance(cal_df, pd.DataFrame) or len(cal_df) == 0:
                    continue

                needed = {"p_adj", "LCL", "UCL"}
                if not needed.issubset(set(cal_df.columns)):
                    raise ValueError(
                        f"Calibration df for var '{var_name}' part '{part_name}' must include {sorted(list(needed))}."
                    )

                p_adj = pd.to_numeric(cal_df["p_adj"], errors="coerce")
                L = pd.to_numeric(cal_df["LCL"], errors="coerce")
                U = pd.to_numeric(cal_df["UCL"], errors="coerce")

                excludes_1 = (L > 1.0) | (U < 1.0)
                flagged_rows = (p_adj < sig_level) & excludes_1
                any_flag = bool(np.any(flagged_rows.fillna(False).to_numpy()))

                cal_metric = f"Calibration {part_name}"
                if any_flag:
                    keep_cal.append(cal_metric)

                ex = []
                if any_flag:
                    idxs = np.where(flagged_rows.fillna(False).to_numpy())[0]
                    if "Comparison" in cal_df.columns:
                        for j in idxs[:top_k_examples_per_metric]:
                            ex.append(str(cal_df.iloc[int(j)]["Comparison"]))
                    else:
                        ex.append(f"{int(len(idxs))} comparisons flagged")

                rows.append(
                    {
                        "Variable": var_name,
                        "Metric": cal_metric,
                        "Type": "Calibration",
                        "min_pairwise_p_adj_for_metric": np.nan,
                        "Significant": any_flag,
                        "Any_pair_violation": any_flag,
                        "Meets_both": any_flag,
                        "Top_violation_examples": "; ".join(ex),
                    }
                )

            metric_list_by_var.setdefault(var_name, [])
            metric_list_by_var[var_name].extend(keep_cal)

    for k in list(metric_list_by_var.keys()):
        metric_list_by_var[k] = list(dict.fromkeys(metric_list_by_var[k]))

    summary_df = (
        pd.DataFrame(rows)
        .sort_values(
            ["Variable", "Meets_both", "Significant", "Any_pair_violation", "Type", "Metric"],
            ascending=[True, False, False, False, True, True],
        )
        .reset_index(drop=True)
    )

    return summary_df, metric_list_by_var


# ============================================================
# Example usage (fill with your objects)
# ============================================================

_all_tables_by_var = {
    "Race/Ethnicity": race_table,
    "Sex": sex_table,
    "Age": age_table,
    "HPI": hpi_table,
}

_all_group_cols_by_var = {
    "Race/Ethnicity": "race_ethnicity",
    "Sex": "gender_identity",
    "Age": "age_group",
    "HPI": "hpi_quartile",
}

_all_pairwise_p_tables_by_var = {
    "Race/Ethnicity": race_pairwise,
    "Sex": sex_pairwise,
    "Age": age_pairwise,
    "HPI": hpi_pairwise,
}

_all_calibration_tables_by_var = {
    "Race/Ethnicity": {"Intercept": tbl_int_race, "Slope": tbl_slope_race},
    "Sex": {"Intercept": tbl_int_sex, "Slope": tbl_slope_sex},
    "Age": {"Intercept": tbl_int_age, "Slope": tbl_slope_age},
    "HPI": {"Intercept": tbl_int_hpi, "Slope": tbl_slope_hpi},
}

_availability_by_var = {
    "Race/Ethnicity": has_race_ethnicity,
    "Sex": has_gender_identity,
    "Age": has_age_group,
    "HPI": has_hpi_score,
}

# A disparity comparison requires at least one usable pairwise result.
tables_by_var = {
    k: v
    for k, v in _all_tables_by_var.items()
    if (
        _availability_by_var.get(k, False)
        and isinstance(v, pd.DataFrame)
        and not v.empty
        and isinstance(_all_pairwise_p_tables_by_var.get(k), pd.DataFrame)
        and not _all_pairwise_p_tables_by_var[k].empty
    )
}
group_cols_by_var = {k: _all_group_cols_by_var[k] for k in tables_by_var}
pairwise_p_tables_by_var = {
    k: v
    for k, v in _all_pairwise_p_tables_by_var.items()
    if k in tables_by_var and isinstance(v, pd.DataFrame)
}
calibration_tables_by_var = {
    k: v
    for k, v in _all_calibration_tables_by_var.items()
    if k in tables_by_var and isinstance(v, dict)
}

if tables_by_var:
    summary_df, metric_list_by_var = summarize_disparities_all_vars_pairwise_any(
        data=data,
        tables_by_var=tables_by_var,
        group_cols_by_var=group_cols_by_var,
        metric_col="Metric",
        ratio_low=ratio_low,
        ratio_high=ratio_high,
        sig_level=0.05,
        top_k_examples_per_metric=5,
        calibration_tables_by_var=calibration_tables_by_var,
        pairwise_p_tables_by_var=pairwise_p_tables_by_var,
        pairwise_p_col="p_adj",
    )
else:
    summary_df, metric_list_by_var = pd.DataFrame(), {}

In [ ]:
for key, mets in metric_list_by_var.items():
    display(Markdown(f"**{key}**"))
    if len(mets) == 0:
        print("  None.")
    else:
        for m in mets:
            print(f"  - {m}")

# flagged = summary_df.loc[summary_df["Meets_both"]].reset_index(drop=True)
# display(flagged)

## Appendix A - Results of all pairwise comparisons for each attribute group

For each metric and attribute with at least two eligible groups, pairwise comparisons were conducted to identify which groups differed. The Holm method was applied within each metric to control the family-wise Type I error rate at 0.05.

### Race/Ethnicity

#### Calibration Intercept

In [ ]:
# ============================================================
# Formatting helpers: p-values and effect estimates
# ============================================================


def format_p_value(p, *, digits=4, threshold=0.0001, na_str="NA"):
    if pd.isna(p):
        return na_str
    p = float(p)
    if p < threshold:
        return f"<{threshold:.{digits}f}"
    return f"{p:.{digits}f}"


def format_or_ci(or_, lcl, ucl, *, digits=3, na_str="NA", include_ci=True):
    if pd.isna(or_):
        return na_str
    or_ = float(or_)
    if not include_ci:
        return f"{or_:.{digits}f}"
    if pd.isna(lcl) or pd.isna(ucl):
        return f"{or_:.{digits}f}"
    return f"{or_:.{digits}f} [{float(lcl):.{digits}f}, {float(ucl):.{digits}f}]"


def format_calibration_comparison_table(
    tbl_int_race: pd.DataFrame,
    *,
    digits_or=3,
    digits_p=4,
    p_threshold=0.0001,
    comparison_col="Comparison",
    or_col="OR",
    lcl_col="LCL",
    ucl_col="UCL",
    p_col="p",
    p_adj_col="p_adj",
    na_str="NA",
    replace_delim=True,
    include_ci_in_estimate=False,
):
    df = tbl_int_race.copy()
    if comparison_col not in df.columns:
        raise KeyError(f"Missing required column: {comparison_col}")
    if or_col not in df.columns:
        raise KeyError(f"Missing required column: {or_col}")
    for c in [p_col, p_adj_col]:
        if c not in df.columns:
            raise KeyError(f"Missing required column: {c}")
    if include_ci_in_estimate:
        for c in [lcl_col, ucl_col]:
            if c not in df.columns:
                raise KeyError(f"Missing required column: {c}")

    comp = df[comparison_col].astype(str)
    if replace_delim:
        comp = comp.str.replace(r"\s*--\s*", " vs ", regex=True)
    df["Comparison"] = comp

    df["Estimated Odds Ratio"] = [
        format_or_ci(o, l, u, digits=digits_or, na_str=na_str, include_ci=include_ci_in_estimate)
        for o, l, u in zip(
            df[or_col],
            df[lcl_col] if lcl_col in df.columns else [np.nan] * len(df),
            df[ucl_col] if ucl_col in df.columns else [np.nan] * len(df),
        )
    ]

    df["Raw p-value"] = [
        format_p_value(p, digits=digits_p, threshold=p_threshold, na_str=na_str) for p in df[p_col]
    ]
    df["Adjusted p-value"] = [
        format_p_value(p, digits=digits_p, threshold=p_threshold, na_str=na_str)
        for p in df[p_adj_col]
    ]

    return df[["Comparison", "Estimated Odds Ratio", "Raw p-value", "Adjusted p-value"]].copy()


# ============================================================
# Race/Ethnicity: Calibration intercept
# ============================================================
if has_race_ethnicity and isinstance(tbl_int_race, pd.DataFrame):
    calibration_tbl = format_calibration_comparison_table(
        tbl_int_race,
        include_ci_in_estimate=False,
    )
    display(HTML(calibration_tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Calibration intercept pairwise comparison within race/ethnicity (Holm-adjusted).**"
        )
    )
else:
    display(
        Markdown("**Race/ethnicity data not available, skipping calibration intercept table.**")
    )

#### Calibration Slope

In [ ]:
if has_race_ethnicity and isinstance(tbl_slope_race, pd.DataFrame):
    calibration_tbl = format_calibration_comparison_table(
        tbl_slope_race, include_ci_in_estimate=True
    )
    display(HTML(calibration_tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Calibration slope pairwise comparison within race/ethnicity (Holm-adjusted).**"
        )
    )

#### Sensitivity

In [ ]:
# ============================================================
# Formatting helpers: pairwise OR estimates
# ============================================================


def format_logor_to_or(logor, *, digits=3, na_str="NA"):
    if pd.isna(logor):
        return na_str
    return f"{float(np.exp(logor)):.{digits}f}"


def format_pairwise_metric_from_big_table(
    pairwise_all_metrics,
    metric,
    *,
    metric_col="Metric",
    comparison_col="Comparison",
    group_a_col="Group_A",
    group_b_col="Group_B",
    observed_diff_col="Observed_Diff",
    p_col="p",
    p_adj_col="p_adj",
    n_a_col="N_A",
    n_b_col="N_B",
    digits_diff: int = 3,
    digits_p: int = 4,
    p_threshold: float = 0.0001,
    na_str: str = "NA",
    use_adj_p_for_flag: bool = True,
    include_raw_p: bool = True,
    sort_by_counts: bool = True,
    replace_delim: bool = True,
):
    if not isinstance(pairwise_all_metrics, pd.DataFrame) or pairwise_all_metrics.empty:
        base_cols = ["Comparison", "Observed Difference", "Adjusted p-value"]
        if include_raw_p:
            base_cols.insert(2, "Raw p-value")
        return pd.DataFrame(columns=base_cols)

    df = pairwise_all_metrics.copy()
    if metric_col not in df.columns:
        raise KeyError(f"Missing metric column '{metric_col}' in pairwise_all_metrics.")

    d = df.loc[df[metric_col].astype(str) == str(metric)].copy()
    if d.empty:
        base_cols = ["Comparison", "Observed_Diff", "Adjusted p-value"]
        if include_raw_p:
            base_cols.insert(2, "Raw p-value")
        return pd.DataFrame(columns=base_cols)

    if comparison_col in d.columns:
        comp = d[comparison_col].astype(str)
    else:
        missing = [c for c in [group_a_col, group_b_col] if c not in d.columns]
        if missing:
            raise KeyError(
                f"Need '{comparison_col}' or both '{group_a_col}' and '{group_b_col}'. Missing: {missing}"
            )
        comp = d[group_a_col].astype(str) + " -- " + d[group_b_col].astype(str)

    if replace_delim:
        comp = comp.str.replace(r"\s*--\s*", " vs ", regex=True)
    d["Comparison"] = comp

    if observed_diff_col not in d.columns:
        raise KeyError(f"Missing '{observed_diff_col}' column.")

    diff_num = pd.to_numeric(d[observed_diff_col], errors="coerce")
    d["Observed Difference"] = diff_num.map(
        lambda x: na_str if pd.isna(x) else f"{float(x):.{digits_diff}f}"
    )

    if p_col not in d.columns:
        raise KeyError(f"Missing '{p_col}' column.")
    if p_adj_col not in d.columns:
        d[p_adj_col] = np.nan

    p_raw_num = pd.to_numeric(d[p_col], errors="coerce")
    p_adj_num = pd.to_numeric(d[p_adj_col], errors="coerce")

    d["Raw p-value"] = [
        format_p_value(x, digits=digits_p, threshold=p_threshold, na_str=na_str) for x in p_raw_num
    ]
    d["Adjusted p-value"] = [
        format_p_value(x, digits=digits_p, threshold=p_threshold, na_str=na_str) for x in p_adj_num
    ]
    d["p_use"] = p_adj_num if use_adj_p_for_flag else p_raw_num

    if sort_by_counts and (n_b_col in d.columns) and (n_a_col in d.columns):
        d[n_b_col] = pd.to_numeric(d[n_b_col], errors="coerce")
        d[n_a_col] = pd.to_numeric(d[n_a_col], errors="coerce")
        d = d.sort_values([n_b_col, n_a_col], ascending=[False, False]).reset_index(drop=True)

    out_cols = ["Comparison", "Observed Difference"]
    if include_raw_p:
        out_cols.append("Raw p-value")
    out_cols.append("Adjusted p-value")

    return d[out_cols].copy()


# ============================================================
# Race/Ethnicity: Sensitivity pairwise comparisons
# ============================================================
if has_race_ethnicity and isinstance(race_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=race_pairwise,
        metric="Sensitivity",
        use_adj_p_for_flag=True,
        include_raw_p=True,
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Sensitivity pairwise comparison within race/ethnicity (Holm-adjusted).**"
        )
    )
else:
    display(Markdown("**Race/ethnicity data not available, skipping pairwise metric tables.**"))

#### Specificity


In [ ]:
if has_race_ethnicity and isinstance(race_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=race_pairwise, metric="Specificity", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Specificity pairwise comparison within race/ethnicity (Holm-adjusted).**"
        )
    )

#### PPV

In [ ]:
if has_race_ethnicity and isinstance(race_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=race_pairwise, metric="PPV", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: PPV pairwise comparison within race/ethnicity (Holm-adjusted).**"
        )
    )

#### NPV

In [ ]:
if has_race_ethnicity and isinstance(race_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=race_pairwise, metric="NPV", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: NPV pairwise comparison within race/ethnicity (Holm-adjusted).**"
        )
    )

#### Accuracy

In [ ]:
if has_race_ethnicity and isinstance(race_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=race_pairwise, metric="Accuracy", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Accuracy pairwise comparison within race/ethnicity (Holm-adjusted).**"
        )
    )

#### FPR

In [ ]:
if has_race_ethnicity and isinstance(race_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=race_pairwise, metric="FPR", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: FPR pairwise comparison within race/ethnicity (Holm-adjusted).**"
        )
    )

#### FNR

In [ ]:
if has_race_ethnicity and isinstance(race_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=race_pairwise, metric="FNR", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: FNR pairwise comparison within race/ethnicity (Holm-adjusted).**"
        )
    )

#### FDR

In [ ]:
if has_race_ethnicity and isinstance(race_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=race_pairwise, metric="FDR", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: FDR pairwise comparison within race/ethnicity (Holm-adjusted).**"
        )
    )

#### FOR

In [ ]:
if has_race_ethnicity and isinstance(race_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=race_pairwise, metric="FOR", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: FOR pairwise comparison within race/ethnicity (Holm-adjusted).**"
        )
    )

#### AUROC

In [ ]:
if has_race_ethnicity and isinstance(race_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=race_pairwise, metric="AUROC", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: AUROC pairwise comparison within race/ethnicity (Holm-adjusted).**"
        )
    )

#### AUPRC

In [ ]:
if has_race_ethnicity and isinstance(race_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=race_pairwise, metric="AUPRC", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: AUPRC pairwise comparison within race/ethnicity (Holm-adjusted).**"
        )
    )

### Sex

#### Calibration Intercept

In [ ]:
if has_gender_identity and isinstance(tbl_int_sex, pd.DataFrame):
    calibration_tbl = format_calibration_comparison_table(tbl_int_sex, include_ci_in_estimate=True)
    display(HTML(calibration_tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Calibration intercept pairwise comparison within sex (Holm-adjusted).**"
        )
    )
else:
    display(Markdown("**Sex/gender data not available, skipping calibration intercept table.**"))

#### Calibration Slope

In [ ]:
if has_gender_identity and isinstance(tbl_slope_sex, pd.DataFrame):
    calibration_tbl = format_calibration_comparison_table(
        tbl_slope_sex, include_ci_in_estimate=True
    )
    display(HTML(calibration_tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Calibration slope pairwise comparison within sex (Holm-adjusted).**"
        )
    )

#### Sensitivity

In [ ]:
if has_gender_identity and isinstance(sex_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=sex_pairwise,
        metric="Sensitivity",
        use_adj_p_for_flag=True,
        include_raw_p=True,
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Sensitivity pairwise comparison within sex (Holm-adjusted).**"
        )
    )
else:
    display(Markdown("**Sex/gender data not available, skipping pairwise metric tables.**"))

#### Specificity


In [ ]:
if has_gender_identity and isinstance(sex_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=sex_pairwise, metric="Specificity", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Specificity pairwise comparison within sex (Holm-adjusted).**"
        )
    )

#### PPV

In [ ]:
if has_gender_identity and isinstance(sex_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=sex_pairwise, metric="PPV", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(f"**Table {iteration_tbl}: PPV pairwise comparison within sex (Holm-adjusted).**")
    )

#### NPV

In [ ]:
if has_gender_identity and isinstance(sex_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=sex_pairwise, metric="NPV", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(f"**Table {iteration_tbl}: NPV pairwise comparison within sex (Holm-adjusted).**")
    )

#### Accuracy

In [ ]:
if has_gender_identity and isinstance(sex_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=sex_pairwise, metric="Accuracy", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Accuracy pairwise comparison within sex (Holm-adjusted).**"
        )
    )

#### FPR

In [ ]:
if has_gender_identity and isinstance(sex_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=sex_pairwise, metric="FPR", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(f"**Table {iteration_tbl}: FPR pairwise comparison within sex (Holm-adjusted).**")
    )

#### FNR

In [ ]:
if has_gender_identity and isinstance(sex_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=sex_pairwise, metric="FNR", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(f"**Table {iteration_tbl}: FNR pairwise comparison within sex (Holm-adjusted).**")
    )

#### FDR

In [ ]:
if has_gender_identity and isinstance(sex_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=sex_pairwise, metric="FDR", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(f"**Table {iteration_tbl}: FDR pairwise comparison within sex (Holm-adjusted).**")
    )

#### FOR

In [ ]:
if has_gender_identity and isinstance(sex_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=sex_pairwise, metric="FOR", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(f"**Table {iteration_tbl}: FOR pairwise comparison within sex (Holm-adjusted).**")
    )

#### AUROC

In [ ]:
if has_gender_identity and isinstance(sex_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=sex_pairwise, metric="AUROC", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: AUROC pairwise comparison within sex (Holm-adjusted).**"
        )
    )

#### AUPRC

In [ ]:
if has_gender_identity and isinstance(sex_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=sex_pairwise, metric="AUPRC", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: AUPRC pairwise comparison within sex (Holm-adjusted).**"
        )
    )

### Age

#### Calibration Intercept

In [ ]:
if has_age_group and isinstance(tbl_int_age, pd.DataFrame):
    calibration_tbl = format_calibration_comparison_table(tbl_int_age, include_ci_in_estimate=True)
    display(HTML(calibration_tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Calibration intercept pairwise comparison within age (Holm-adjusted).**"
        )
    )
else:
    display(Markdown("**Age data not available, skipping calibration intercept table.**"))

#### Calibration Slope

In [ ]:
if has_age_group and isinstance(tbl_slope_age, pd.DataFrame):
    calibration_tbl = format_calibration_comparison_table(
        tbl_slope_age, include_ci_in_estimate=True
    )
    display(HTML(calibration_tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Calibration slope pairwise comparison within age (Holm-adjusted).**"
        )
    )

#### Sensitivity

In [ ]:
if has_age_group and isinstance(age_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=age_pairwise,
        metric="Sensitivity",
        use_adj_p_for_flag=True,
        include_raw_p=True,
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Sensitivity pairwise comparison within age (Holm-adjusted).**"
        )
    )
else:
    display(Markdown("**Age data not available, skipping pairwise metric tables.**"))

#### Specificity


In [ ]:
if has_age_group and isinstance(age_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=age_pairwise, metric="Specificity", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Specificity pairwise comparison within age (Holm-adjusted).**"
        )
    )

#### PPV

In [ ]:
if has_age_group and isinstance(age_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=age_pairwise, metric="PPV", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(f"**Table {iteration_tbl}: PPV pairwise comparison within age (Holm-adjusted).**")
    )

#### NPV

In [ ]:
if has_age_group and isinstance(age_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=age_pairwise, metric="NPV", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(f"**Table {iteration_tbl}: NPV pairwise comparison within age (Holm-adjusted).**")
    )

#### Accuracy

In [ ]:
if has_age_group and isinstance(age_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=age_pairwise, metric="Accuracy", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Accuracy pairwise comparison within age (Holm-adjusted).**"
        )
    )

#### FPR

In [ ]:
if has_age_group and isinstance(age_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=age_pairwise, metric="FPR", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(f"**Table {iteration_tbl}: FPR pairwise comparison within age (Holm-adjusted).**")
    )

#### FNR

In [ ]:
if has_age_group and isinstance(age_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=age_pairwise, metric="FNR", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(f"**Table {iteration_tbl}: FNR pairwise comparison within age (Holm-adjusted).**")
    )

#### FDR

In [ ]:
if has_age_group and isinstance(age_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=age_pairwise, metric="FDR", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(f"**Table {iteration_tbl}: FDR pairwise comparison within age (Holm-adjusted).**")
    )

#### FOR

In [ ]:
if has_age_group and isinstance(age_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=age_pairwise, metric="FOR", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(f"**Table {iteration_tbl}: FOR pairwise comparison within age (Holm-adjusted).**")
    )

#### AUROC

In [ ]:
if has_age_group and isinstance(age_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=age_pairwise, metric="AUROC", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: AUROC pairwise comparison within age (Holm-adjusted).**"
        )
    )

#### AUPRC

In [ ]:
if has_age_group and isinstance(age_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=age_pairwise, metric="AUPRC", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: AUPRC pairwise comparison within age (Holm-adjusted).**"
        )
    )

### HPI

#### Calibration Intercept

In [ ]:
if has_hpi_score and isinstance(tbl_int_hpi, pd.DataFrame):
    calibration_tbl = format_calibration_comparison_table(tbl_int_hpi, include_ci_in_estimate=True)
    display(HTML(calibration_tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Calibration intercept pairwise comparison within HPI (Holm-adjusted).**"
        )
    )
else:
    display(Markdown("**HPI data not available, skipping calibration intercept table.**"))

#### Calibration Slope

In [ ]:
if has_hpi_score and isinstance(tbl_slope_hpi, pd.DataFrame):
    calibration_tbl = format_calibration_comparison_table(
        tbl_slope_hpi, include_ci_in_estimate=True
    )
    display(HTML(calibration_tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Calibration slope pairwise comparison within HPI (Holm-adjusted).**"
        )
    )

#### Sensitivity

In [ ]:
if has_hpi_score and isinstance(hpi_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=hpi_pairwise,
        metric="Sensitivity",
        use_adj_p_for_flag=True,
        include_raw_p=True,
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Sensitivity pairwise comparison within HPI (Holm-adjusted).**"
        )
    )
else:
    display(Markdown("**HPI data not available, skipping pairwise metric tables.**"))

#### Specificity


In [ ]:
if has_hpi_score and isinstance(hpi_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=hpi_pairwise, metric="Specificity", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Specificity pairwise comparison within HPI (Holm-adjusted).**"
        )
    )

#### PPV

In [ ]:
if has_hpi_score and isinstance(hpi_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=hpi_pairwise, metric="PPV", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(f"**Table {iteration_tbl}: PPV pairwise comparison within HPI (Holm-adjusted).**")
    )

#### NPV

In [ ]:
if has_hpi_score and isinstance(hpi_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=hpi_pairwise, metric="NPV", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(f"**Table {iteration_tbl}: NPV pairwise comparison within HPI (Holm-adjusted).**")
    )

#### Accuracy

In [ ]:
if has_hpi_score and isinstance(hpi_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=hpi_pairwise, metric="Accuracy", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: Accuracy pairwise comparison within HPI (Holm-adjusted).**"
        )
    )

#### FPR

In [ ]:
if has_hpi_score and isinstance(hpi_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=hpi_pairwise, metric="FPR", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(f"**Table {iteration_tbl}: FPR pairwise comparison within HPI (Holm-adjusted).**")
    )

#### FNR

In [ ]:
if has_hpi_score and isinstance(hpi_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=hpi_pairwise, metric="FNR", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(f"**Table {iteration_tbl}: FNR pairwise comparison within HPI (Holm-adjusted).**")
    )

#### FDR

In [ ]:
if has_hpi_score and isinstance(hpi_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=hpi_pairwise, metric="FDR", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(f"**Table {iteration_tbl}: FDR pairwise comparison within HPI (Holm-adjusted).**")
    )

#### FOR

In [ ]:
if has_hpi_score and isinstance(hpi_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=hpi_pairwise, metric="FOR", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(f"**Table {iteration_tbl}: FOR pairwise comparison within HPI (Holm-adjusted).**")
    )

#### AUROC

In [ ]:
if has_hpi_score and isinstance(hpi_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=hpi_pairwise, metric="AUROC", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: AUROC pairwise comparison within HPI (Holm-adjusted).**"
        )
    )

#### AUPRC

In [ ]:
if has_hpi_score and isinstance(hpi_pairwise, pd.DataFrame):
    tbl = format_pairwise_metric_from_big_table(
        pairwise_all_metrics=hpi_pairwise, metric="AUPRC", use_adj_p_for_flag=True
    )
    display(HTML(tbl.to_html(index=False)))
    iteration_tbl += 1
    display(
        Markdown(
            f"**Table {iteration_tbl}: AUPRC pairwise comparison within HPI (Holm-adjusted).**"
        )
    )